# v25_rl

Kaggriculture submission using the v25(`local_arena\v25_rl\v25_rl.py`) crop, animal, and market planner with the deterministic PPO worker policy from `Co_Kaggle\g5\local_arena\v25_rl\runs\worker_ppo_static_v20_v13_plant_signal\checkpoints\best.pt`.

Source checkpoint SHA-256: `aede6be8c7a4d1057489b6e9ed761514b9dc3af5b457f37574f8aa9970c29a55`. The exact actor float32 weights are embedded in `main.py`. The submitted agent uses only the Python standard library.


## 1. Install the same environment version used for local validation

This installation is only for building/testing the notebook. The submitted `main.py` itself has no external dependencies.

In [1]:
%%capture
!pip install --upgrade "kaggle-environments==1.32.7"

## 2. Write the required v25_rl entrypoint

This single file contains the planner, the PPO worker policy, and the trained actor weights. The superseded worker scheduler, critic, and training reward evaluator are omitted.


In [2]:
%%writefile main.py
"""Kaggriculture v25_rl submission; standard-library runtime and embedded PPO actor."""
import base64
import hashlib
import sys
import types
import zlib

# The policy sources stay readable and self-contained in this one file.
_REWARD_SOURCE = r'''
"""Inference helpers for the v25 trained worker policy."""
from __future__ import annotations

CROP_PRODUCTS = ("WHEAT", "CARROT", "TOMATO", "STRAWBERRY", "MELON")
ANIMAL_PRODUCT_NAMES = ("MILK", "EGG", "WOOL")
PRODUCTS = CROP_PRODUCTS + ANIMAL_PRODUCT_NAMES
ANIMAL_PRODUCTS = {"COW": "MILK", "GOOSE": "EGG", "SHEEP": "WOOL"}
CROP_DECAY_AGE = {"WHEAT": 5, "CARROT": 4, "TOMATO": 12, "STRAWBERRY": 17, "MELON": 13}
OPS = ("PLANT", "DIG", "HARVEST", "WATER", "FERTILIZE", "FEED", "CARE", "COLLECT_FERTILIZER", "BUILD_COOP", "BUILD_PASTURE", "PLACE_ANIMAL", "PICKUP", "DELIVER", "PASS")
ITEMS = ("", "WHEAT", "CARROT", "TOMATO", "STRAWBERRY", "MELON", "COW", "SHEEP", "GOOSE", "MILK", "EGG", "WOOL", "FERTILIZER")
MOVE_ACTIONS = {"NORTH", "SOUTH", "EAST", "WEST"}
GAME_LAST_ACTION_STEP = 29 * 24 + 23


def plant_can_deliver_before_end(executor, obs, crop, target, travel=0):
    """Conservative time budget for first legal HARVEST then shed delivery.

    Workers restart at sheds each day. Reserve shed-to-crop travel on the
    maturity day, HARVEST, return travel and PLACE. This uses the engine's
    first-yield age, including partial early WHEAT/CARROT yields.
    Capacity/watering admission is checked separately by WorkerPolicy.
    """
    first = getattr(executor, "CROP_FIRST_YIELD_DAY", {}).get(crop)
    if first is None:
        spec = getattr(executor, "CROPS", {}).get(crop)
        if not spec:
            return False
        first = min(age for age, _ in spec[2])
    plant_day = (int(obs["day"]) * 24 + int(obs["hour"]) + int(travel)) // 24
    sheds = getattr(executor, "SHED", ())
    distance = min((abs(target[0]-p[0])+abs(target[1]-p[1]) for p in sheds), default=0)
    delivery_step = (plant_day + int(first)) * 24 + 2 * distance + 1
    return delivery_step <= GAME_LAST_ACTION_STEP


def _positions(obs) -> list[tuple[int, int]]:
    farm = obs["farms"][obs["player"]]
    return [tuple(farm["farmer"])] + [tuple(p) for p in farm["hands"]]


def _crop_decay_start_step(executor, tile) -> int | None:
    raw_value = tile.get("max_lifespan_step", -1)
    raw = int(raw_value if raw_value is not None else -1)
    if raw >= 0:
        return raw
    crop = tile.get("crop")
    if crop in CROP_DECAY_AGE:
        return (int(tile.get("planted_day", 0) or 0) + CROP_DECAY_AGE[crop]) * 24
    spec = executor.CROPS.get(crop)
    if not spec:
        return None
    events = tuple(spec[2] or ())
    if not events:
        return None
    final_age = max(int(age) for age, _units in events)
    planted = int(tile.get("planted_day", 0) or 0)
    return (planted + final_age + 1) * 24
'''

_POLICY_SOURCE = r'''
"""RL worker policy that replaces v25_rl.unit_actions completely."""
from __future__ import annotations
from collections import defaultdict
from dataclasses import dataclass, field, replace
from typing import Any
from math import ceil, tanh
from operator import mul
import struct
from worker_reward import ANIMAL_PRODUCTS, ITEMS, MOVE_ACTIONS, OPS, PRODUCTS, _positions, plant_can_deliver_before_end, _crop_decay_start_step as crop_decay_start_step

@dataclass(frozen=True)
class Task:
    target: tuple[int,int] | None
    op: str
    item: str=""
    amount: int=1
    critical: float=0.0
    slot: int=0
    planned: bool=False
    deadline_step: int | None=None
    release_step: int=0
    @property
    def key(self): return (self.target,self.op,self.item,self.slot)

GLOBAL_FEATURE_NAMES=("day","hour","workers","shed_fill","wheat","fertilizer","products","seeds","animals","crops","weeds","unfed","critical_unfed","unwatered","critical_unwatered","yield_units","carried_products","crop_plan","animal_plan","free_workers")
BASE_FEATURE_NAMES=("day","hour","workers","free_workers","worker_x","worker_y","target_x","target_y","distance","at_target","worker_inv","worker_wheat","worker_fert","worker_products","worker_animals","shed_fill","shed_wheat","shed_fert","seed_count","age","yield_units","consecutive_unwatered","consecutive_unfed","watered","fed","cared","fert_days","care_bonus","fert_available","critical","amount","shed_distance")
CANDIDATE_FEATURE_NAMES=BASE_FEATURE_NAMES+tuple("op_"+x for x in OPS)+tuple("item_"+(x or "NONE") for x in ITEMS)

# PASS/defer and non-plant planner shaping below are measured directly in
# normalized-advantage units.  Planned PLANT is intentionally different: its
# completion credit is expressed in raw reward-equivalent units so it can be
# calibrated against an ordinary crop HARVEST before PPO normalization.
PLANNED_PLANT_REWARD_EQUIV = 16.0
# v12's +16 divided by rollout GAE std (~420) supplied only ~0.04.
# Add a modest scale-independent preference after confirmed useful completion.
PLANNED_PLANT_COMPLETION_ACTOR_BONUS = 0.25
PLANNED_BUILD_ACTOR_BONUS = 1.0
PLANNED_ANIMAL_PICKUP_ACTOR_BONUS = 1.0
PLANNED_PLACE_ANIMAL_ACTOR_BONUS = 1.5
DEFER_PLANNED_PLANT_ACTOR_PENALTY = -0.75
AVOIDABLE_PASS_ACTOR_PENALTY = -1.0

def _weed_prevention_task(task):
    return task.op=="WATER" and float(task.critical)>=1.0

def _animal_survival_task(task):
    return (
        (task.op=="FEED" and float(task.critical)>=1.0)
        or (
            task.op in ("PICKUP","HARVEST")
            and task.item=="WHEAT"
            and float(task.critical)>=1.0
        )
    )

def _animal_setup_task(task):
    return task.planned and (
        task.op in ("BUILD_COOP","BUILD_PASTURE","PLACE_ANIMAL")
        or (task.op in ("PICKUP","DIG") and task.item in ANIMAL_PRODUCTS)
    )

def _crop_decay_start_step(e,tile):
    return crop_decay_start_step(e,tile)

def _crop_decay_deadline_step(e,obs,tile):
    """Harvest/retire before the first decay, independent of held quantity.

    The engine applies worker actions before decay at the same absolute step,
    so HARVEST at decay_start itself is still lossless.
    """
    start=_crop_decay_start_step(e,tile)
    if start is None:
        return None
    # Ongoing crops remain on the tile after HARVEST. Leave an action for DIG
    # before the engine turns an exhausted zero-yield plant into a weed.
    if tile.get("crop") in ("TOMATO","STRAWBERRY"):
        # Hands disappear and the farmer returns to the shed at midnight.
        # Finish both HARVEST and DIG using today's workforce, before reset.
        return start-(2 if float(tile.get("yield_units",0) or 0)>0 else 1)
    return start


def _crop_salvage_deadline_step(e,obs,tile):
    """Last action step that can still collect positive yield after spoilage."""
    start=_crop_decay_start_step(e,tile)
    units=int(ceil(float(tile.get("yield_units",0) or 0)))
    if start is None or units<=0:
        return None
    now=int(obs["day"])*24+int(obs["hour"])
    next_decay=start if now<=start else now+(now-start)%2
    return next_decay+2*(units-1)

def totals(private):
    out=defaultdict(int)
    for k,v in private["shed"].items(): out[k]+=int(v)
    for inv in private["inventories"]:
        for k,v in inv.items(): out[k]+=int(v)
    return out

def global_features(e,obs,animal_plan,crop_plan,free_workers):
    farm=obs["farms"][obs["player"]]; p=obs["private"]; t=totals(p)
    animals=crops=weeds=unfed=cunfed=unwatered=cunwatered=0; held=0.0
    for row in farm["tiles"]:
        for x in row:
            if not isinstance(x,dict): continue
            if x.get("animal"):
                animals+=1; held+=float(x.get("yield_units",0) or 0)
                if not x.get("fed_today"):
                    unfed+=1; cunfed+=int(int(x.get("consecutive_unfed",0) or 0)>=1)
            elif x.get("kind")=="PLANT":
                crops+=1; held+=float(x.get("yield_units",0) or 0)
                if not x.get("watered_today"):
                    unwatered+=1; cunwatered+=int(int(x.get("consecutive_unwatered",0) or 0)>=1)
            elif x.get("kind")=="WEED": weeds+=1
    carried=sum(int(inv.get(q,0) or 0) for inv in p["inventories"] for q in PRODUCTS)
    v=[obs["day"]/29,obs["hour"]/23,len(p["inventories"])/16,sum(p["shed"].values())/100,t.get("WHEAT",0)/40,t.get("FERTILIZER",0)/40,sum(t.get(q,0) for q in PRODUCTS)/150,sum(p["seeds"].values())/50,animals/20,crops/50,weeds/30,unfed/20,cunfed/20,unwatered/50,cunwatered/50,held/100,carried/100,len(crop_plan)/100,len(animal_plan)/20,free_workers/16]
    return tuple(max(-5.0,min(5.0,float(x))) for x in v)

def candidate_features(e,obs,worker,task,free_workers):
    farm=obs["farms"][obs["player"]]; p=obs["private"]; pos=_positions(obs)[worker]; inv=p["inventories"][worker]
    target=task.target if task.target is not None else tuple(e.nearest_shed(pos))
    tile=e.tile(farm,task.target) if task.target is not None else {}
    tile=tile if isinstance(tile,dict) else {}
    age=obs["day"]-int(tile.get("planted_day",tile.get("placed_day",obs["day"])))
    base=[obs["day"]/29,obs["hour"]/23,len(p["inventories"])/16,free_workers/16,pos[0]/9,pos[1]/9,target[0]/9,target[1]/9,e.dist(pos,target)/18,float(pos==target),sum(inv.values())/30,int(inv.get("WHEAT",0))/10,int(inv.get("FERTILIZER",0))/10,sum(int(inv.get(q,0)) for q in PRODUCTS)/30,sum(int(inv.get(a,0)) for a in e.ANIMALS)/5,sum(p["shed"].values())/100,int(p["shed"].get("WHEAT",0))/40,int(p["shed"].get("FERTILIZER",0))/40,int(p["seeds"].get(task.item,0))/20,age/30,float(tile.get("yield_units",0) or 0)/12,float(tile.get("consecutive_unwatered",0) or 0)/2,float(tile.get("consecutive_unfed",0) or 0)/2,float(bool(tile.get("watered_today"))),float(bool(tile.get("fed_today"))),float(bool(tile.get("cared_today"))),(int(tile.get("fertilized_until_day",-1))-obs["day"])/10,float(tile.get("pending_care_bonus",0) or 0)/8,float(bool(tile.get("fertilizer_available"))),task.critical,task.amount/10,e.dist(target,e.nearest_shed(target))/18]
    oh=[0.0]*len(OPS); oh[OPS.index(task.op)]=1.0
    ih=[0.0]*len(ITEMS); ih[ITEMS.index(task.item) if task.item in ITEMS else 0]=1.0
    return tuple(max(-5.0,min(5.0,float(x))) for x in base+oh+ih)

class ActorModel:
    """Exact float32 checkpoint coefficients, evaluated with Python's math module."""

    def __init__(self, actor_bytes, expected_dim):
        raw=actor_bytes
        magic,dim,hidden=struct.unpack_from("<4sII",raw)
        if magic!=b"AR25" or dim!=expected_dim or hidden!=128:
            raise ValueError("Incompatible v25 actor weights")
        expected_count=hidden*dim+hidden+hidden*hidden+hidden+hidden+1
        if len(raw)!=12+4*expected_count:
            raise ValueError("Incomplete v25 actor weights")
        values=struct.unpack_from(f"<{expected_count}f",raw,12)
        offset=0
        def rows(count,width):
            nonlocal offset
            output=tuple(values[offset+i*width:offset+(i+1)*width] for i in range(count))
            offset+=count*width
            return output
        def vector(count):
            nonlocal offset
            output=values[offset:offset+count]
            offset+=count
            return output
        self.w0=rows(hidden,dim)
        self.b0=vector(hidden)
        self.w2=rows(hidden,hidden)
        self.b2=vector(hidden)
        self.w4=vector(hidden)
        self.b4=vector(1)[0]

    def logits(self,candidates):
        output=[]
        w0,b0,w2,b2,w4,b4=self.w0,self.b0,self.w2,self.b2,self.w4,self.b4
        for candidate in candidates:
            h1=[tanh(sum(map(mul,row,candidate),bias)) for row,bias in zip(w0,b0)]
            h2=[tanh(sum(map(mul,row,h1),bias)) for row,bias in zip(w2,b2)]
            output.append(sum(map(mul,w4,h2),b4))
        return output


@dataclass
class SubDecision:
    candidates: tuple
    action_index: int
    old_log_prob: float
    actor_bonus: float = 0.0
    reward_equiv_bonus: float = 0.0
    # Bookkeeping only: this amount is already included in actor_bonus.
    plant_completion_actor_bonus: float = 0.0

@dataclass
class TurnRecord:
    state: tuple
    subdecisions: list[SubDecision]
    old_value: float
    turn: int
    reward: float=0.0
    reward_breakdown: dict=field(default_factory=dict)
    advantage: float=0.0
    return_target: float=0.0

class WorkerPolicy:
    def __init__(self,executor,model,device,deterministic=False,collect=True,rollout_temperature=0.2):
        if rollout_temperature<=0:
            raise ValueError("rollout_temperature must be > 0")
        self.e=executor; self.model=model; self.device=device; self.deterministic=deterministic; self.collect=collect
        self.rollout_temperature=float(rollout_temperature)
        self.records=[]; self.pending=None; self.candidate_counts=[]
        # A policy decision selects a job, not a one-turn direction. Workers keep
        # moving toward that job until it completes or becomes invalid.
        self.active_tasks: dict[int, Task] = {}
        # Originating PPO sample for a remote planned route.  Positive planner
        # shaping is credited to this original choice only when the route
        # reaches its execution point; interrupted/invalid routes earn nothing.
        self.active_origins: dict[int, SubDecision] = {}
        self.active_day: int | None = None
        # Track WHEAT that came from the shed so returning the same resource to
        # the shed cannot masquerade as newly completed production logistics.
        self.shed_sourced_wheat: dict[int, int] = defaultdict(int)
        self.turn_delivery_credit: list[dict[str, int]] = []
        # Reward-only provenance for planner execution, useful route movement,
        # and PASS choices that skipped feasible work.
        self.turn_plan_credit: list[dict[str, str]] = []
        self.turn_planned_plant_origins: list[tuple[int,SubDecision]] = []
        self.turn_route_progress: list[bool] = []
        self.turn_avoidable_pass: list[bool] = []

    def tasks(self,obs,animal_plan,crop_plan):
        e=self.e; farm=obs["farms"][obs["player"]]; p=obs["private"]; day=obs["day"]; tasks=[]; need=defaultdict(int)
        for pt,planned in crop_plan.items():
            pt=tuple(pt); t=e.tile(farm,pt)
            if t is None:
                if int(p["seeds"].get(planned,0)): tasks.append(Task(pt,"PLANT",planned,planned=True))
                continue
            if not isinstance(t,dict): continue
            if t.get("kind")=="WEED": tasks.append(Task(pt,"DIG")); continue
        # Planner changes must not orphan assets already on the farm.
        live_plants=[((x,y),t) for y,row in enumerate(farm["tiles"]) for x,t in enumerate(row)]
        for pt,t in live_plants:
            if not isinstance(t,dict) or t.get("kind")!="PLANT": continue
            planned=t.get("crop","")
            crop=t.get("crop",planned); age=day-int(t.get("planted_day",day)); spec=e.CROPS.get(crop)
            first_yield_age=int(getattr(e,"CROP_FIRST_YIELD_DAY",{}).get(crop,10**9))
            # HARVEST legality comes from the engine's first_yield_day, not
            # from the planner's yield schedule. WHEAT/CARROT are legal at age
            # 2 even though the planner schedules their nominal yield later.
            if float(t.get("yield_units",0) or 0)>0 and age>=first_yield_age:
                tasks.append(Task(
                    pt,"HARVEST",crop,
                    deadline_step=_crop_decay_deadline_step(e,obs,t),
                ))
            if day<29 and not t.get("watered_today"): tasks.append(Task(pt,"WATER",crop,critical=float(int(t.get("consecutive_unwatered",0) or 0)>=1)))
            useful=max(a for a,_ in spec[2]) if spec and crop in ("TOMATO","STRAWBERRY") else (int(spec[3]) if spec else 0)
            if day<29 and age<=useful and int(t.get("fertilized_until_day",-1))<day: tasks.append(Task(pt,"FERTILIZE","FERTILIZER"))
            exhausted=(age>=useful if crop in ("TOMATO","STRAWBERRY") else age>useful)
            if exhausted and float(t.get("yield_units",0) or 0)<=0:
                tasks.append(Task(pt,"DIG",deadline_step=_crop_decay_deadline_step(e,obs,t)))
        for pt,a in animal_plan.items():
            pt=tuple(pt); t=e.tile(farm,pt); structure="COOP" if a=="GOOSE" else "PASTURE"
            if t is None: tasks.append(Task(pt,"BUILD_COOP" if a=="GOOSE" else "BUILD_PASTURE",a,planned=True))
            elif isinstance(t,dict) and t.get("kind")=="WEED": tasks.append(Task(pt,"DIG",a,planned=True))
            elif isinstance(t,dict) and t.get("kind") in ("COOP","PASTURE") and not t.get("animal"):
                if t.get("kind")==structure: tasks.append(Task(pt,"PLACE_ANIMAL",a,planned=True)); need[a]+=1
                else: tasks.append(Task(pt,"DIG",a,planned=True))
        unfed=0; critical_unfed=0; critical_feed_targets=[]
        for y,row in enumerate(farm["tiles"]):
            for x,t in enumerate(row):
                if not isinstance(t,dict) or not t.get("animal"): continue
                pt=(x,y); a=t["animal"]
                if day<29 and not t.get("fed_today"):
                    is_critical=int(t.get("consecutive_unfed",0) or 0)>=1
                    tasks.append(Task(
                        pt,"FEED","WHEAT",critical=float(is_critical)
                    ))
                    unfed+=1
                    critical_unfed+=int(is_critical)
                    if is_critical:
                        critical_feed_targets.append(pt)
                if day<29 and not t.get("cared_today"): tasks.append(Task(pt,"CARE",a))
                if float(t.get("yield_units",0) or 0)>0: tasks.append(Task(pt,"HARVEST",ANIMAL_PRODUCTS.get(a,"")))
                if t.get("fertilizer_available"): tasks.append(Task(pt,"COLLECT_FERTILIZER","FERTILIZER"))

        def pickups(item,short,cap,critical_units=0):
            left=min(short,int(p["shed"].get(item,0)))
            critical_left=min(max(0,int(critical_units)),left)
            slot=0
            while left>0:
                n=min(cap,left)
                tasks.append(Task(
                    None,"PICKUP",item,n,
                    critical=float(critical_left>0),
                    slot=slot,
                ))
                left-=n
                critical_left=max(0,critical_left-n)
                slot+=1

        carried_wheat=sum(
            int(inv.get("WHEAT",0) or 0) for inv in p["inventories"]
        )
        remaining=max(0,23-int(obs["hour"]))
        positions=_positions(obs)
        reachable_critical_wheat=0
        for w,pos in enumerate(positions):
            units=int(p["inventories"][w].get("WHEAT",0) or 0)
            if units<=0:
                continue
            if any(
                e.dist(pos,target)<=remaining
                for target in critical_feed_targets
            ):
                reachable_critical_wheat+=units
        ordinary_wheat_short=max(0,unfed-carried_wheat)
        critical_wheat_short=max(
            0,critical_unfed-reachable_critical_wheat
        )
        # Frozen market orders may leave the shed empty. Mature field WHEAT
        # is also a feed prerequisite; reserve its HARVEST before ordinary work.
        if critical_wheat_short>int(p["shed"].get("WHEAT",0)):
            tasks=[replace(t,critical=1.0) if t.op=="HARVEST" and t.item=="WHEAT" else t for t in tasks]
        # Stranded carried WHEAT must not suppress an emergency shed pickup.
        # Generate at least enough pickup capacity to cover the critical
        # reachable-supply deficit, even when total carried WHEAT would make
        # the ordinary aggregate shortage appear to be zero.
        wheat_short=max(ordinary_wheat_short,critical_wheat_short)
        pickups(
            "WHEAT",wheat_short,4,
            critical_units=critical_wheat_short,
        )
        pickups("FERTILIZER",max(0,sum(t.op=="FERTILIZE" for t in tasks)-sum(int(i.get("FERTILIZER",0)) for i in p["inventories"])),3)
        for a,n in need.items():
            shortage=max(0,n-sum(int(i.get(a,0)) for i in p["inventories"]))
            for slot in range(min(shortage,int(p["shed"].get(a,0)))):
                tasks.append(Task(None,"PICKUP",a,1,slot=slot,planned=True))
        return tasks

    def _sync_day(self,obs):
        day=int(obs["day"])
        if self.active_day != day:
            self.active_day=day
            self.active_tasks.clear()
            self.active_origins.clear()
            self.shed_sourced_wheat.clear()

    def _delivery_credit_for_worker(self,obs,w):
        inv=obs["private"]["inventories"][w]
        out={}
        for q in PRODUCTS:
            n=max(0,int(inv.get(q,0) or 0))
            if q=="WHEAT":
                n=max(0,n-int(self.shed_sourced_wheat.get(w,0)))
            out[q]=n
        return out

    def extras(self,obs,w):
        pos=_positions(obs)[w]; inv=obs["private"]["inventories"][w]; shed=tuple(self.e.nearest_shed(pos)); out=[]
        credit=self._delivery_credit_for_worker(obs,w)
        for q in PRODUCTS:
            n=int(credit.get(q,0) or 0)
            if n: out.append(Task(shed,"DELIVER",q,n))
        out.append(Task(pos,"PASS"))
        return out

    def _track_resource_action(self,w,action):
        if not action:
            return
        op=action[0]
        if op=="PICKUP" and len(action)>=2 and action[1]=="WHEAT":
            n=int(action[2]) if len(action)>=3 else 1
            self.shed_sourced_wheat[w]+=max(0,n)
        elif op=="FEED" and self.shed_sourced_wheat.get(w,0)>0:
            self.shed_sourced_wheat[w]-=1

    def feasible(self,obs,w,t,seeds,shed):
        e=self.e; inv=obs["private"]["inventories"][w]; pos=_positions(obs)[w]; target=t.target if t.target is not None else tuple(e.nearest_shed(pos))
        distance=e.dist(pos,target)
        remaining=max(0,23-int(obs["hour"]))
        # A newly planted seed starts with consecutive_unwatered=1 and becomes
        # WEED if it cannot be watered before the day refresh. Reserve one full
        # later turn for WATER after movement + PLANT.
        if t.op=="PLANT":
            if distance+1>remaining: return False
        elif (
            t.op in ("PICKUP","HARVEST")
            and t.item=="WHEAT"
            and float(t.critical)>=1.0
        ):
            # Emergency WHEAT supply is useful only if this worker can reach
            # the source, PICKUP/HARVEST, then reach at least one critical
            # animal and FEED it before the end-of-day escape refresh.
            farm=obs["farms"][obs["player"]]
            critical_animals=[]
            for y,row in enumerate(farm["tiles"]):
                for x,tile in enumerate(row):
                    if (
                        isinstance(tile,dict)
                        and tile.get("animal")
                        and not tile.get("fed_today")
                        and int(tile.get("consecutive_unfed",0) or 0)>=1
                    ):
                        critical_animals.append((x,y))
            if not critical_animals:
                return False
            followup=min(e.dist(target,pt) for pt in critical_animals)
            if distance+1+followup>remaining: return False
        elif distance>remaining:
            return False
        if t.op=="HARVEST" and t.deadline_step is not None:
            # Missing the first-loss deadline makes salvage urgent, not illegal.
            tile=e.tile(obs["farms"][obs["player"]],target)
            salvage=_crop_salvage_deadline_step(e,obs,tile)
            if salvage is not None and int(obs["day"])*24+int(obs["hour"])+distance>salvage:
                return False
        if t.op=="PLANT" and int(seeds.get(t.item,0))<=0: return False
        if t.op=="PLANT" and not self._plant_has_capacity(obs,w,t,seeds,shed): return False
        if t.op=="FEED" and int(inv.get("WHEAT",0))<=0: return False
        if t.op=="FERTILIZE" and int(inv.get("FERTILIZER",0))<=0: return False
        if t.op=="PLACE_ANIMAL" and int(inv.get(t.item,0))<=0: return False
        if t.op=="PLACE_ANIMAL" and not self._animal_has_capacity(obs,w,t,shed): return False
        if t.op=="PICKUP" and int(shed.get(t.item,0))<=0: return False
        if t.op=="DELIVER" and int(inv.get(t.item,0))<=0: return False
        return True

    def emit(self,obs,w,t):
        e=self.e; pos=_positions(obs)[w]; target=t.target if t.target is not None else tuple(e.nearest_shed(pos))
        if pos!=target: return e.move(pos,target)
        if t.op=="PLANT": return ["PLANT",t.item]
        if t.op=="PLACE_ANIMAL": return ["PLACE",t.item]
        if t.op=="PICKUP": return ["PICKUP",t.item,t.amount]
        if t.op=="DELIVER": return ["PLACE",t.item,t.amount]
        if t.op=="PASS": return ["PASS"]
        return [t.op]

    def planned_completion_bonus(self,t):
        """Normalized actor-only completion bonus for non-plant plan steps."""
        if not t.planned:
            return 0.0
        if t.op in ("BUILD_COOP","BUILD_PASTURE"):
            return PLANNED_BUILD_ACTOR_BONUS
        if t.op=="PICKUP" and t.item in self.e.ANIMALS:
            return PLANNED_ANIMAL_PICKUP_ACTOR_BONUS
        if t.op=="PLACE_ANIMAL":
            return PLANNED_PLACE_ANIMAL_ACTOR_BONUS
        return 0.0

    def planned_completion_reward_equiv(self,t):
        """Raw reward-equivalent credit, normalized later with rollout GAE."""
        if t.planned and t.op=="PLANT":
            return PLANNED_PLANT_REWARD_EQUIV
        return 0.0

    def avoidable_plant_delay(self,w,t,choices):
        # Survival, maintenance, harvest/delivery and the forced animal pipeline
        # are legitimate reasons to defer planting. Alternatives here already
        # passed seed, travel and shared maintenance admission. Only useful
        # planting alternatives receive a deferral preference.
        if t.op in ("PLANT","WATER","FEED","CARE","HARVEST","DELIVER"):
            return False
        if _animal_setup_task(t) or (t.op=="DIG" and t.deadline_step is not None):
            return False
        if _animal_survival_task(t):
            return False
        obs=getattr(self,"_choice_obs",None)
        return any(cw==w and ct.planned and ct.op=="PLANT" and (
            obs is None or plant_can_deliver_before_end(
                self.e,obs,ct.item,ct.target,self.e.dist(_positions(obs)[w],ct.target)
            )
        ) for cw,ct,_ in choices)

    def actor_bonus_for_choice(self,w,t,choices,completed=False):
        same_worker=[ct for cw,ct,_ in choices if cw==w]
        bonus=0.0
        if t.op=="PASS" and any(ct.op!="PASS" for ct in same_worker):
            bonus+=AVOIDABLE_PASS_ACTOR_PENALTY
        if self.avoidable_plant_delay(w,t,choices):
            bonus+=DEFER_PLANNED_PLANT_ACTOR_PENALTY
        if completed:
            bonus+=self.planned_completion_bonus(t)
        return float(bonus)

    def _capacity_states(self,obs):
        now=int(obs["day"])*24+int(obs["hour"])
        return {
            w:(pos,now,int(obs["private"]["inventories"][w].get("WHEAT",0)))
            for w,pos in enumerate(_positions(obs))
        }

    def _advance_capacity_state(self,obs,t,state,shed_wheat,allow_pickup=False):
        """Project travel + action, including the FEED wheat prerequisite."""
        pos,ready,wheat=state
        target=t.target if t.target is not None else tuple(self.e.nearest_shed(pos))
        used=0
        if t.op=="FEED" and wheat<=0:
            if not allow_pickup or shed_wheat<=0:
                return None
            shed_pos=tuple(self.e.nearest_shed(pos))
            ready+=self.e.dist(pos,shed_pos)+1
            pos=shed_pos
            wheat+=1
            used=1
        action_step=max(ready+self.e.dist(pos,target),int(t.release_step))
        if t.op=="FEED":
            wheat-=1
        elif t.op=="PICKUP" and t.item=="WHEAT":
            if t.amount>shed_wheat:
                return None
            used=t.amount
            wheat+=t.amount
        elif t.op=="HARVEST" and t.item=="WHEAT":
            tile=self.e.tile(obs["farms"][obs["player"]],target)
            if isinstance(tile,dict):
                wheat+=int(tile.get("yield_units",0) or 0)
        ready=action_step+1
        if t.op=="HARVEST" and t.item in ("TOMATO","STRAWBERRY"):
            tile=self.e.tile(obs["farms"][obs["player"]],target)
            start=_crop_decay_start_step(self.e,tile) if isinstance(tile,dict) else None
            if start is not None and action_step>=start-24:
                ready+=1  # final harvest must be followed by exhausted cleanup
        return (target,ready,wheat),used,action_step

    def _deadline_schedule(self,obs,tasks,states=None,shed_wheat=None,delay=0):
        """Bounded EDF route estimate; a worker may finish several jobs.

        This is a conservative capacity guard, not an optimal-routing proof.
        It accounts for movement, each action, shared wheat, and release times.
        Unlike one-task-per-worker matching it exposes queued harvest pressure.
        """
        states=dict(self._capacity_states(obs) if states is None else states)
        states={w:(p,ready+delay,wheat) for w,(p,ready,wheat) in states.items()}
        stock=int(obs["private"]["shed"].get("WHEAT",0)) if shed_wheat is None else shed_wheat
        end=int(obs["day"])*24+23
        routes={w:[] for w in states}
        completed=set()
        slack={w:float("inf") for w in states}
        # Stable order, with survival first on equal deadlines. Planning each
        # action from its predecessor reserves cumulative worker time.
        def compatible_workers(t):
            deadline=end if t.deadline_step is None else min(end,t.deadline_step)
            return sum(
                (p:=self._advance_capacity_state(obs,t,state,stock,allow_pickup=True)) is not None
                and p[2]<=deadline for state in states.values()
            )
        def first_completion(t):
            finishes=[p[2] for state in states.values() if
                (p:=self._advance_capacity_state(obs,t,state,stock,allow_pickup=True)) is not None]
            return min(finishes,default=float("inf"))
        ordered=sorted(tasks,key=lambda t:(
            end if t.deadline_step is None else min(end,t.deadline_step),
            compatible_workers(t),
            0 if _weed_prevention_task(t) or _animal_survival_task(t) else 1,
            first_completion(t),
            repr(t.key),
        ))
        for t in ordered:
            deadline=end if t.deadline_step is None else min(end,int(t.deadline_step))
            candidates=[]
            for w,state in states.items():
                projected=self._advance_capacity_state(obs,t,state,stock,allow_pickup=True)
                if projected is not None and projected[2]<=deadline:
                    candidates.append((projected[2],w,projected))
            if not candidates:
                continue
            _,w,(new_state,used,action_step)=min(candidates,key=lambda c:(c[0],c[1]))
            stock-=used
            states[w]=new_state
            routes[w].append(t)
            slack[w]=min(slack[w],deadline-action_step)
            completed.add(t.key)
        return completed,routes,slack

    def _maintenance_tasks(self,obs,tasks):
        end=int(obs["day"])*24+23
        return [t for t in tasks if (
            t.op in ("WATER","FEED")
            or (t.op in ("HARVEST","DIG") and t.deadline_step is not None and t.deadline_step<=end+1)
        )]

    def _plant_has_capacity(self,obs,w,t,seeds,shed):
        now=int(obs["day"])*24+int(obs["hour"])
        in_turn=getattr(self,"_capacity_step",None)==now
        states=dict(self.turn_capacity_states if in_turn else self._capacity_states(obs))
        tasks=self.turn_capacity_tasks if in_turn else self.tasks(obs,{}, {})
        reserved=self.turn_capacity_reserved if in_turn else set()
        proposed=list(self.turn_capacity_plants if in_turn else [])
        projected=self._advance_capacity_state(obs,t,states[w],int(shed.get("WHEAT",0)))
        if projected is None:
            return False
        states[w]=projected[0]
        water=Task(t.target,"WATER",t.item,critical=1.0,release_step=projected[0][1])
        proposed.append(water)
        jobs=[job for job in self._maintenance_tasks(obs,tasks) if job.key not in reserved]+proposed
        stock=int(shed.get("WHEAT",0))
        if t.item=="WHEAT":
            # Do not deadlock the feed-production pipeline by forbidding its
            # renewal when today's wheat is depleted. Reserve realistic pickup
            # time; actual FEED still requires observed wheat in feasible().
            stock=max(stock,sum(job.op=="FEED" for job in jobs))
        covered,_,_=self._deadline_schedule(obs,jobs,states,stock)
        if len(covered)<len({job.key for job in jobs}):
            return False
        animals=list(self.turn_capacity_animals if in_turn else [])
        return self._next_day_maintenance_fits(obs,states,proposed,animals)

    def _animal_has_capacity(self,obs,w,t,shed):
        now=int(obs["day"])*24+int(obs["hour"])
        in_turn=getattr(self,"_capacity_step",None)==now
        states=dict(self.turn_capacity_states if in_turn else self._capacity_states(obs))
        tasks=self.turn_capacity_tasks if in_turn else self.tasks(obs,{}, {})
        reserved=self.turn_capacity_reserved if in_turn else set()
        plants=list(self.turn_capacity_plants if in_turn else [])
        animals=list(self.turn_capacity_animals if in_turn else [])
        projected=self._advance_capacity_state(obs,t,states[w],int(shed.get("WHEAT",0)))
        if projected is None:
            return False
        states[w]=projected[0]
        jobs=[job for job in self._maintenance_tasks(obs,tasks) if job.key not in reserved]+plants
        covered,_,_=self._deadline_schedule(obs,jobs,states,int(shed.get("WHEAT",0)))
        if len(covered)<len({job.key for job in jobs}):
            return False
        # New animals start consecutive_unfed=0: unlike PLANT, PLACE does not
        # require feeding on its placement day to survive. Reserve tomorrow's
        # FEED/CARE workload, including other placements assigned this turn.
        animals.append(t)
        return self._next_day_maintenance_fits(obs,states,plants,animals)

    def _next_day_maintenance_fits(self,obs,states,plants,animals):
        # Reserve a full day's WATER, FEED and CARE for the enlarged farm.
        # Use the current workforce; future market hires are not guaranteed.
        if int(obs["day"])>=29:
            return True
        tomorrow={**obs,"day":int(obs["day"])+1,"hour":0}
        daily=[]
        for y,row in enumerate(obs["farms"][obs["player"]]["tiles"]):
            for x,tile in enumerate(row):
                if not isinstance(tile,dict):
                    continue
                if tile.get("kind")=="PLANT":
                    daily.append(Task((x,y),"WATER",tile.get("crop","")))
                elif tile.get("animal"):
                    daily.extend((Task((x,y),"FEED","WHEAT"),Task((x,y),"CARE",tile["animal"])))
        daily.extend(Task(job.target,"WATER",job.item) for job in plants)
        for job in animals:
            daily.extend((Task(job.target,"FEED","WHEAT"),Task(job.target,"CARE",job.item)))
        # Reserve pickup time per future FEED. Future supply is unknown and may
        # include today's planting/harvest; using today's stock as a hard future
        # limit would forbid WHEAT planting precisely when feed needs renewal.
        future_states={q:(p,int(tomorrow["day"])*24,0) for q,(p,_,_) in states.items()}
        feed_jobs=sum(job.op=="FEED" for job in daily)
        covered,_,_=self._deadline_schedule(tomorrow,daily,future_states,feed_jobs)
        return len(covered)==len({job.key for job in daily})

    def _urgent_harvest_tasks(self,obs,worker_ids,tasks):
        """Reserve cumulative travel/action time before the first crop loss."""
        if not worker_ids:
            return []
        now=int(obs["day"])*24+int(obs["hour"])
        positions=_positions(obs)
        urgent=[]
        end=int(obs["day"])*24+23
        due=[t for t in tasks if t.op in ("HARVEST","DIG") and t.deadline_step is not None and t.deadline_step<=end+1]
        critical=[t for t in tasks if _weed_prevention_task(t) or (t.op=="FEED" and _animal_survival_task(t))]
        states={w:self._capacity_states(obs)[w] for w in worker_ids}
        covered,routes,slack=self._deadline_schedule(obs,critical+due,states)
        delayed,_,_=self._deadline_schedule(obs,critical+due,states,delay=1)
        pressured={t.key for w,route in routes.items() if slack[w]<=1 for t in route if t.op in ("HARVEST","DIG")}
        if len(delayed)<len(covered):
            pressured.update(t.key for route in routes.values() for t in route if t.op in ("HARVEST","DIG"))
        for t in tasks:
            if t.op not in ("HARVEST","DIG") or t.deadline_step is None or t.target is None:
                continue
            distances=[self.e.dist(positions[w],t.target) for w in worker_ids]
            if not distances:
                continue
            # One-turn reserve: when waiting one more turn would consume the
            # last safe start opportunity, reserve a worker now.
            # An exhausted ongoing plant has no remaining production benefit.
            # Retire it promptly instead of creating another midnight rescue.
            final_harvest=self._retirement_after_harvest(obs,t) is not None
            if t.op=="DIG" or final_harvest or t.key in pressured or now+min(distances)+1>=int(t.deadline_step):
                urgent.append(t)
        return urgent

    def _retirement_after_harvest(self,obs,t):
        """Commit cleanup once the ongoing crop has produced its final yield."""
        if t.op!="HARVEST" or t.item not in ("TOMATO","STRAWBERRY") or t.target is None:
            return None
        tile=self.e.tile(obs["farms"][obs["player"]],t.target)
        start=_crop_decay_start_step(self.e,tile) if isinstance(tile,dict) else None
        now=int(obs["day"])*24+int(obs["hour"])
        if start is None or now<start-24:
            return None
        return Task(t.target,"DIG",deadline_step=start-1)

    def _max_critical_task_matching(self,obs,worker_ids,tasks,seeds,shed):
        """Maximum reachable assignments for hard-deadline scheduler tasks."""
        if not tasks or not worker_ids:
            return 0
        feasible_by_task=[
            [
                w for w in worker_ids
                if self.feasible(obs,w,t,seeds,shed)
            ]
            for t in tasks
        ]
        order=sorted(range(len(tasks)),key=lambda i:len(feasible_by_task[i]))
        worker_match={}

        def augment(task_i,seen):
            for w in feasible_by_task[task_i]:
                if w in seen:
                    continue
                seen.add(w)
                prev=worker_match.get(w)
                if prev is None or augment(prev,seen):
                    worker_match[w]=task_i
                    return True
            return False

        matched=0
        for task_i in order:
            if augment(task_i,set()):
                matched+=1
        return matched

    def unit_actions(self,obs,animal_plan,crop_plan):
        self._sync_day(obs)
        self._choice_obs=obs
        positions=_positions(obs)
        workers=list(range(len(positions)))
        actions=[None]*len(workers)
        tasks=self.tasks(obs,animal_plan,crop_plan)
        seeds=dict(obs["private"]["seeds"])
        shed=dict(obs["private"]["shed"])
        critical_water_tasks=[t for t in tasks if _weed_prevention_task(t)]
        critical_feed_tasks=[
            t for t in tasks
            if t.op=="FEED" and float(t.critical)>=1.0
        ]
        critical_wheat_supply_tasks=[
            t for t in tasks
            if (
                t.op in ("PICKUP","HARVEST")
                and t.item=="WHEAT"
                and float(t.critical)>=1.0
            )
        ]
        direct_survival_tasks=critical_water_tasks+critical_feed_tasks
        survival_tasks=direct_survival_tasks+critical_wheat_supply_tasks
        deadline_harvest_tasks=self._urgent_harvest_tasks(
            obs,workers,tasks
        )
        critical_tasks=survival_tasks+deadline_harvest_tasks
        direct_survival_keys={t.key for t in direct_survival_tasks}
        survival_supply_keys={t.key for t in critical_wheat_supply_tasks}
        deadline_harvest_keys={t.key for t in deadline_harvest_tasks}
        critical_keys={t.key for t in critical_tasks}
        reserved=set()
        self._capacity_step=int(obs["day"])*24+int(obs["hour"])
        self.turn_capacity_states=self._capacity_states(obs)
        self.turn_capacity_tasks=tasks
        self.turn_capacity_reserved=reserved
        self.turn_capacity_plants=[]
        self.turn_capacity_animals=[]
        # Snapshot provenance before this turn's actions. Reward attribution uses
        # this to cap delivery credit to goods that did not originate in the shed.
        self.turn_delivery_credit=[
            self._delivery_credit_for_worker(obs,w) for w in workers
        ]
        self.turn_plan_credit=[{} for _ in workers]
        self.turn_planned_plant_origins=[]
        self.turn_route_progress=[False for _ in workers]
        self.turn_avoidable_pass=[False for _ in workers]
        self.turn_avoidable_plant_delay=[False for _ in workers]


        def reserve_resources(t):
            if t.op=="PLANT":
                seeds[t.item]=max(0,int(seeds.get(t.item,0))-1)
            elif t.op=="PICKUP":
                shed[t.item]=max(0,int(shed.get(t.item,0))-t.amount)

        def reserve_capacity(w,t):
            projected=self._advance_capacity_state(obs,t,self.turn_capacity_states[w],int(shed.get("WHEAT",0)))
            if projected is not None:
                self.turn_capacity_states[w]=projected[0]
                if t.op=="PLANT":
                    self.turn_capacity_plants.append(Task(
                        t.target,"WATER",t.item,critical=1.0,release_step=projected[0][1]
                    ))
                elif t.op=="PLACE_ANIMAL":
                    self.turn_capacity_animals.append(t)

        def record_reward_provenance(w,t,pos,target,action,avoidable_pass=False):
            if t.planned and pos==target and t.op in ("PLANT","PLACE_ANIMAL"):
                self.turn_plan_credit[w]={"op":t.op,"item":t.item}
            useful_route=t.op!="PLANT" or plant_can_deliver_before_end(
                self.e,obs,t.item,target,self.e.dist(pos,target)
            )
            if useful_route and t.op!="PASS" and pos!=target and action and action[0] in MOVE_ACTIONS:
                self.turn_route_progress[w]=True
            if t.op=="PASS" and avoidable_pass:
                self.turn_avoidable_pass[w]=True

        def assign_committed(w,t,reservation_key):
            pos=positions[w]
            target=t.target if t.target is not None else tuple(self.e.nearest_shed(pos))
            action=self.emit(obs,w,t)
            actions[w]=action
            record_reward_provenance(w,t,pos,target,action)
            reserved.add(reservation_key)
            reserve_capacity(w,t)
            reserve_resources(t)
            # Only actual resource operations change provenance; movement does not.
            if pos==target:
                origin=self.active_origins.pop(w,None)
                if origin is not None:
                    origin.actor_bonus+=self.planned_completion_bonus(t)
                    if t.planned and t.op=="PLANT":
                        self.turn_planned_plant_origins.append((w,origin))
                    else:
                        origin.reward_equiv_bonus+=self.planned_completion_reward_equiv(t)
                self._track_resource_action(w,action)
                cleanup=self._retirement_after_harvest(obs,t)
                if cleanup is None:
                    self.active_tasks.pop(w,None)
                else:
                    self.active_tasks[w]=cleanup
            else:
                self.active_tasks[w]=t

        # First honor valid routes chosen on earlier turns. A committed route does
        # not create a new PPO actor sample: the actor already chose this job when
        # the route started.
        for w in list(workers):
            active=self.active_tasks.get(w)
            if active is None:
                continue
            # Preserve a non-critical committed route unless this worker is
            # required for lexicographic hard-deadline coverage:
            #   1) immediate survival (critical WATER / FEED)
            #   2) survival resource prerequisite (critical WHEAT PICKUP)
            #   3) imminent crop-decay HARVEST
            # This prevents harvest protection from sacrificing animals and
            # still preempts only the minimum number of unrelated routes.
            if active.key not in critical_keys:
                remaining_direct=[
                    t for t in direct_survival_tasks if t.key not in reserved
                ]
                remaining_supply=[
                    t for t in critical_wheat_supply_tasks if t.key not in reserved
                ]
                remaining_harvest=[
                    t for t in deadline_harvest_tasks if t.key not in reserved
                ]
                other_workers=[q for q in workers if q!=w]
                if remaining_direct or remaining_supply or remaining_harvest:
                    with_direct=self._max_critical_task_matching(
                        obs,workers,remaining_direct,seeds,shed
                    )
                    without_direct=self._max_critical_task_matching(
                        obs,other_workers,remaining_direct,seeds,shed
                    )
                    with_supply=self._max_critical_task_matching(
                        obs,workers,remaining_direct+remaining_supply,seeds,shed
                    )
                    without_supply=self._max_critical_task_matching(
                        obs,other_workers,remaining_direct+remaining_supply,seeds,shed
                    )
                    with_total=self._max_critical_task_matching(
                        obs,workers,
                        remaining_direct+remaining_supply+remaining_harvest,
                        seeds,shed
                    )
                    without_total=self._max_critical_task_matching(
                        obs,other_workers,
                        remaining_direct+remaining_supply+remaining_harvest,
                        seeds,shed
                    )
                    if (
                        without_direct < with_direct
                        or (
                            without_direct == with_direct
                            and without_supply < with_supply
                        )
                        or (
                            without_direct == with_direct
                            and without_supply == with_supply
                            and without_total < with_total
                        )
                    ):
                        self.active_tasks.pop(w,None)
                        self.active_origins.pop(w,None)
                        continue
            candidates=tasks+self.extras(obs,w)
            current=next((t for t in candidates if t.key==active.key),None)
            if current is None or not self.feasible(obs,w,current,seeds,shed):
                self.active_tasks.pop(w,None)
                self.active_origins.pop(w,None)
                continue
            reservation_key=(w,current.key) if current.op=="DELIVER" else current.key
            if reservation_key in reserved:
                self.active_tasks.pop(w,None)
                self.active_origins.pop(w,None)
                continue
            assign_committed(w,current,reservation_key)
            workers.remove(w)

        # Uncommitted workers receive new PPO task assignments.
        while workers:
            choices=[]; feats=[]
            for w in workers:
                for t in tasks+self.extras(obs,w):
                    reservation_key=(w,t.key) if t.op=="DELIVER" else t.key
                    if t.op!="PASS" and reservation_key in reserved:
                        continue
                    if not self.feasible(obs,w,t,seeds,shed):
                        continue
                    choices.append((w,t,reservation_key))
                    feats.append(candidate_features(self.e,obs,w,t,len(workers)))

            # Survival comes before production preservation. First force
            # critical WATER/FEED, then any WHEAT pickup needed to make
            # emergency FEED possible, and only then imminent decay-HARVEST.
            urgent=[
                i for i,(_w,t,_key) in enumerate(choices)
                if t.key in direct_survival_keys
            ]
            if not urgent:
                urgent=[
                    i for i,(_w,t,_key) in enumerate(choices)
                    if t.key in survival_supply_keys
                ]
            if not urgent:
                urgent=[
                    i for i,(_w,t,_key) in enumerate(choices)
                    if t.key in deadline_harvest_keys
                ]
            if urgent:
                choices=[choices[i] for i in urgent]
                feats=[feats[i] for i in urgent]
                # Keep PPO's urgent worker/task choice only when it preserves
                # the reachable deadline coverage. A greedy actor must not
                # give a flexible worker the sole job of a specialist worker.
                remaining=[job for job in direct_survival_tasks+deadline_harvest_tasks if job.key not in reserved]
                baseline,_,_=self._deadline_schedule(obs,remaining,self.turn_capacity_states,int(shed.get("WHEAT",0)))
                safe=[]
                for i,(w,t,_key) in enumerate(choices):
                    projected=self._advance_capacity_state(obs,t,self.turn_capacity_states[w],int(shed.get("WHEAT",0)))
                    if projected is None:
                        continue
                    states=dict(self.turn_capacity_states)
                    states[w]=projected[0]
                    pending=[job for job in remaining if job.key!=t.key]
                    covered,_,_=self._deadline_schedule(obs,pending,states,int(shed.get("WHEAT",0))-projected[1])
                    completed=covered|({t.key} if t.key in {job.key for job in remaining} else set())
                    if baseline<=completed:
                        safe.append(i)
                if safe:
                    choices=[choices[i] for i in safe]
                    feats=[feats[i] for i in safe]

            if not urgent:
                # Finish the planner's animal pipeline before ordinary actor
                # work. Preserve today's reachable maintenance/deadlines when
                # considering setup movement, clearing, BUILD and PICKUP too.
                maintenance=[job for job in self._maintenance_tasks(obs,tasks)
                             if job.key not in reserved]+self.turn_capacity_plants
                stock=int(shed.get("WHEAT",0))
                baseline,_,_=self._deadline_schedule(obs,maintenance,self.turn_capacity_states,stock)
                setup=[]
                for i,(w,t,_key) in enumerate(choices):
                    if not _animal_setup_task(t):
                        continue
                    projected=self._advance_capacity_state(obs,t,self.turn_capacity_states[w],stock)
                    if projected is None:
                        continue
                    states=dict(self.turn_capacity_states)
                    states[w]=projected[0]
                    covered,_,_=self._deadline_schedule(obs,maintenance,states,stock-projected[1])
                    if baseline<=covered:
                        setup.append(i)
                if setup:
                    # Prefer completing a placement, then supplying it, before
                    # starting more structures. PPO still assigns workers/jobs.
                    rank={"PLACE_ANIMAL":0,"PICKUP":1,"BUILD_COOP":2,"BUILD_PASTURE":2,"DIG":3}
                    stage=min(rank[choices[i][1].op] for i in setup)
                    setup=[i for i in setup if rank[choices[i][1].op]==stage]
                    choices=[choices[i] for i in setup]
                    feats=[feats[i] for i in setup]

            # PASS remains valid for workers with no feasible unreserved job.
            # Mask it per worker, so one busy worker does not hide another's
            # genuinely necessary idle action. PPO samples the filtered set.
            busy={w for w,t,_ in choices if t.op!="PASS"}
            keep=[i for i,(w,t,_key) in enumerate(choices) if t.op!="PASS" or w not in busy]
            choices=[choices[i] for i in keep]
            feats=[feats[i] for i in keep]

            if not choices:
                for w in workers:
                    actions[w]=["PASS"]
                    self.active_tasks.pop(w,None)
                break

            raw_logits=self.model.logits(feats)
            j=max(range(len(raw_logits)),key=raw_logits.__getitem__)
            w,t,reservation_key=choices[j]
            pos=positions[w]
            target=t.target if t.target is not None else tuple(self.e.nearest_shed(pos))
            action=self.emit(obs,w,t)
            actions[w]=action
            avoidable_pass=(
                t.op=="PASS"
                and any(cw==w and ct.op!="PASS" for cw,ct,_ in choices)
            )
            record_reward_provenance(w,t,pos,target,action,avoidable_pass)
            self.turn_avoidable_plant_delay[w]=self.avoidable_plant_delay(w,t,choices)
            self.candidate_counts.append(len(choices))

            workers.remove(w)
            if t.op!="PASS":
                reserved.add(reservation_key)
            reserve_capacity(w,t)
            reserve_resources(t)

            if t.op!="PASS" and pos!=target:
                self.active_tasks[w]=t
                self.active_origins.pop(w,None)
            else:
                cleanup=self._retirement_after_harvest(obs,t) if pos==target else None
                if cleanup is None:
                    self.active_tasks.pop(w,None)
                else:
                    self.active_tasks[w]=cleanup
                self.active_origins.pop(w,None)
                if pos==target:
                    self._track_resource_action(w,action)

        return [a or ["PASS"] for a in actions]
'''

_PLANNER_SOURCE = r'''

"""Kaggriculture v25_rl planning and market policy for PPO workers."""


import math

# Crop tuple format: (seed_cost, reference/base market price, yield_schedule, final_yield_age).
# yield_schedule contains (crop_age_in_days, units_produced) events; e.g. WHEAT
# costs 10, has reference price 25, yields 4 units at age 4, and its last yield is age 4.
CROPS={
 'WHEAT':(10,25,((4,4),),4),
 'CARROT':(20,35,((3,3),),3),
 'TOMATO':(50,60,((8,2),(9,2),(10,2),(11,2)),11),
 'STRAWBERRY':(100,120,((10,2),(12,2),(14,2),(16,2)),16),
 'MELON':(80,250,((10,6),),10)}

# Official Kaggriculture engine HARVEST legality thresholds.
# These are intentionally separate from the planner's yield schedule above:
# WHEAT and CARROT can legally be harvested at age 2 even though the planner
# schedules their maximum/nominal yield later.
CROP_FIRST_YIELD_DAY={
 'WHEAT':2,
 'CARROT':2,
 'TOMATO':8,
 'STRAWBERRY':10,
 'MELON':10,
}

ANIMALS={'COW':(400,'MILK',8,2,3),'SHEEP':(500,'WOOL',6,3,4),'GOOSE':(300,'EGG',4,1,2)}

# Per-shop demand on each town consumption tick.
# Values are {product: units_consumed}. Single-product shops consume 2x.
SHOPS={
 'BAKERY':{'WHEAT':1,'EGG':1},
 'PIZZA_SHOP':{'MILK':1,'TOMATO':1,'WHEAT':1},
 'BRUNCH_SPOT':{'EGG':1,'WHEAT':1,'STRAWBERRY':1},
 'YARN_STORE':{'WOOL':2},
 'ICE_CREAM_SHOP':{'STRAWBERRY':1,'MILK':1,'WHEAT':1},
 'PET_CAFE':{'CARROT':2},
 'SMOOTHIE_SHOP':{'STRAWBERRY':1,'MILK':1},
 'FARMERS_MARKET':{'WHEAT':1,'CARROT':1,'TOMATO':1,'STRAWBERRY':1}}

# 17 animal tiles: 6 NW, 7 NE, 4 SW, and none in the lower-right quadrant.
ROUTES=(((4,4),(4,3),(4,2)),((3,4),(3,3),(2,4)),
        ((5,4),(5,3),(5,2)),((6,4),(6,3),(7,4),(6,2)),
        ((4,5),(3,5)),((4,6),(3,6)))
ANIMAL_POINTS={p for route in ROUTES for p in route}
ANIMAL_QUADRANT={p:('S' if p[1]>=5 else 'N')+('E' if p[0]>=5 else 'W')
                 for p in ANIMAL_POINTS}
ANIMAL_QUADRANT_ORDER={'NW':0,'NE':1,'SW':2}
# The shed occupies the central 2x2 tiles. These coordinates are the transfer interface
# used for PICKUP/DROP/PLACE decisions and as routing targets via nearest_shed().
SHED=((4,4),(5,4),(4,5),(5,5))
# The official environment defaults the non-seed shed capacity to 100.
SHED_CAPACITY=100
# Preserve v20_3's sell priority while making the sellable product universe explicit.
SELLABLE_PRODUCTS=('MILK','WOOL','STRAWBERRY','MELON','EGG','TOMATO','CARROT','FERTILIZER','WHEAT')
PASS=['PASS']
# Maximum number of animals the planner should place on route tiles.
HERD_LIMIT=17 # empirical evident from the number of herds of winners in v20 loss cases
# Fill the six NW animal route positions during days 1 and 2 after the opening.
EARLY_NW_ANIMAL_TARGETS={'COW':3,'SHEEP':1,'GOOSE':2}
# Preserve working capital while signal-driven procurement buys new producers.
PROCUREMENT_CASH_RESERVE=100

# Latest in-game day on which a new crop seed may be purchased. These gates prevent
# late-season shortages from buying seeds that cannot mature before the season ends.
LATEST_BUY_DAY={
    'WHEAT':24,
    'CARROT':25,
    'TOMATO':20,
    'STRAWBERRY':18,
    'MELON':18,
}

OPP_STYLE=None
# Legacy opponent-style tuning tuple. In this v20 file it is never read, so its values
# have no runtime effect and the tuple-component semantics cannot be recovered from v20.
TRADER_MIX=(6,4,5)
# Legacy opponent-style tuning tuple; defined but unused in v20, so changing it does
# not change the policy.
NORMAL_MIX=(8,5,0)
# Legacy V16-specific tuning tuple; defined but never consumed in v20. Active V16
# behavior instead comes from OPP_STYLE, V16_FINAL_HANDS, V16_FINAL_BONUS, and seed ordering.
V16_MIX=(7,4,0)
V16_FINAL_HANDS=10
V16_FINAL_BONUS=160

# Global game-clock constants for the 30-day, 24-turn/day season.
TURNS_PER_DAY=24
SEASON_TURNS=720
SHOP_INTERVAL=4
CENTER_INTERVAL=24
SHOP_UNLOCK_TURNS=3*TURNS_PER_DAY
MAX_SHOPS=8

# Opponent hidden-shed prior. Keep these global so ablations can change the
# assumption without touching forecast logic.
OPP_SHED_RATIO=0.8
OPP_SHED_FALLBACK=2

# Simple forecast-only WHEAT reserve heuristic. Each player's daily herd demand is
# modeled as visible herd size + this reserve allowance.
HERD_WHEAT_RESERVE=3

# def price(item, inventory):
#     # Heuristic market-price model used by the crop/animal planners. Inventory 10,000 is
#     # treated as the equilibrium point (d == 0). Scarcity (d < 0) raises the estimate;
#     # surplus (d >= 0) lowers it with item-specific curves, always floored at 1. The
#     # function forecasts planning value from projected inventory; it does not read the
#     # live market price directly.
#     d=inventory-10000
#     if item=='WHEAT':return max(1,25-5*math.log1p(d)/math.log(401)) if d>=0 else 25+math.sqrt(-d)
#     if item in ('CARROT','TOMATO','EGG'):
#         base,through,up,down={'CARROT':(35,450,1,.7),'TOMATO':(60,200,.4,.6),'EGG':(50,332,.4,.2)}[item]
#         u=abs(d)/through
#         if d<0:return base*(1+up*(u+8*max(0,u-1)**2))
#         return max(1,base*(1-down*math.sqrt(u))) if item!='EGG' else max(1,base-10*math.log1p(d)/math.log(333))
#     if item=='STRAWBERRY':return max(1,120-1.92*d) if d>=0 else 120+8.4*math.sqrt(-d)
#     if item=='MELON':return max(1,250-.01*d*d) if d>=0 else 250+50*math.log1p(-d)/math.log(301)
#     if item=='MILK':return max(1,160-256*d/122) if d>=0 else 160+96*math.sqrt(-d/122)
#     if item=='WOOL':return max(1,200-640*(d/105)**2) if d>=0 else 200+40*math.log1p(-d)/math.log(106)
#     return max(1,100-.2*d)

# Manhattan grid distance. It is used for route ordering, nearest-shed selection,
# worker/task travel cost, deadlines, and feasibility checks.
def dist(a,b):return abs(a[0]-b[0])+abs(a[1]-b[1])
def move(a,b):
    # Return one cardinal step from a toward b. Prefer the x-axis when |dx| >= |dy|;
    # otherwise move on y. If already at b, PASS.
    dx=b[0]-a[0];dy=b[1]-a[1]
    if abs(dx)>=abs(dy) and dx:return ['EAST' if dx>0 else 'WEST']
    if dy:return ['SOUTH' if dy>0 else 'NORTH']
    return PASS
def nearest_shed(p):return min(SHED,key=lambda s:dist(p,s))
# Convert an (x, y) farm coordinate to the row-major tile grid entry tiles[y][x].
def tile(f,p):return f['tiles'][p[1]][p[0]]
def totals(private):
    # Aggregate the player's private shed inventory plus every farmer/hand inventory into
    # one item-count dictionary. Seeds and items already placed on farm tiles are separate.
    result=dict(private['shed'])
    for inv in private['inventories']:
        for c,n in inv.items():result[c]=result.get(c,0)+n
    return result

def post_action_market_inventory(obs,actions):
    """Predict market-visible shed contents and sell-reserve totals after worker actions.

    Worker actions execute before market orders. Mirror shed transfers exactly enough for
    SELL availability, and reduce reserve totals for irreversible same-turn consumption
    or loss: FEED consumes one WHEAT, FERTILIZE consumes one FERTILIZER, and DROP overflow
    past shed capacity is discarded. Pure transfers (PICKUP/PLACE/DROP that fits) do not
    change the reserve total.
    """
    f=obs['farms'][obs['player']];p=obs['private']
    positions=[f['farmer']]+f['hands'];invs=p['inventories']
    held=dict(p['shed']);reserve_total=totals(p)

    for i,a in enumerate(actions):
        if i>=len(positions) or i>=len(invs) or not isinstance(a,list) or not a:continue
        pos=tuple(positions[i]);inv=invs[i];op=a[0]

        if op=='PICKUP':
            if pos not in SHED or len(a)<2:continue
            try:n=int(a[2]) if len(a)>=3 else 1
            except (TypeError,ValueError):continue
            if n<=0:continue
            item=a[1];take=min(n,held.get(item,0))
            if take>0:held[item]-=take
            continue

        if op=='DROP':
            if pos not in SHED:continue
            # DROP removes the whole worker inventory; anything beyond shed capacity
            # is destroyed, so subtract that overflow from the reserve total as well.
            for item,n in inv.items():
                if n<=0:continue
                room=max(0,SHED_CAPACITY-sum(held.values()))
                take=min(n,room)
                if take>0:held[item]=held.get(item,0)+take
                lost=n-take
                if lost>0:reserve_total[item]=max(0,reserve_total.get(item,0)-lost)
            continue

        if op=='PLACE':
            if pos not in SHED or len(a)<2:continue
            item=a[1]
            # Animal PLACE on a matching structure is farm placement, not a shed deposit.
            structure='COOP' if item=='GOOSE' else 'PASTURE'
            t=tile(f,pos)
            if item in ANIMALS and isinstance(t,dict) and t.get('kind')==structure and not t.get('animal'):
                continue
            try:n=int(a[2]) if len(a)>=3 else 1
            except (TypeError,ValueError):continue
            if n<=0:continue
            n=min(n,inv.get(item,0))
            room=max(0,SHED_CAPACITY-sum(held.values()))
            n=min(n,room)
            if n>0:held[item]=held.get(item,0)+n
            continue

        if op=='FEED':
            t=tile(f,pos)
            if (inv.get('WHEAT',0)>0 and isinstance(t,dict) and
                    t.get('animal') in ANIMALS and not t.get('fed_today',False)):
                reserve_total['WHEAT']=max(0,reserve_total.get('WHEAT',0)-1)
            continue

        if op=='FERTILIZE':
            t=tile(f,pos)
            if (inv.get('FERTILIZER',0)>0 and isinstance(t,dict) and 'crop' in t and
                    t.get('fertilized_until_day',-1)<obs['day']):
                reserve_total['FERTILIZER']=max(0,reserve_total.get('FERTILIZER',0)-1)

    return held,reserve_total

# =============================================================================
# PRODUCTION SIGNALS
# =============================================================================
# This subsystem converts the currently observable game state into normalized
# crop-production need signals. The helpers below only measure demand, production
# capacity, and shortage; animal purchases use AnimalBuyingPolicy instead.
#
# Internal flow:
#   1) estimate known remaining crop demand;
#   2) estimate remaining output from existing crops and owned seeds;
#   3) assemble demand-capacity gaps and normalize them into comparable scores.


# -----------------------------------------------------------------------------
# Remaining hard-demand model
# -----------------------------------------------------------------------------
def _remaining_hard_demand(obs):
    """Known crop consumption: Town + open shops + observable WHEAT feed."""
    step=obs['day']*TURNS_PER_DAY+obs['hour'];day=obs['day']
    town_ticks=sum(1 for t in range(step,SEASON_TURNS) if t%CENTER_INTERVAL==0)
    shop_ticks=sum(1 for t in range(step,SEASON_TURNS) if t%SHOP_INTERVAL==0)
    shop_per_tick={crop:0. for crop in CROPS}
    for shop in obs['town']['unlocked_shops']:
        for product,units in SHOPS[shop].items():
            if product in shop_per_tick:shop_per_tick[product]+=units
    demand={
        p:float(town_ticks+shop_ticks*shop_per_tick[p])
        for p in CROPS
    }

    # Every visible animal consumes one WHEAT per day. If it has already been fed today,
    # only future days remain; otherwise include today as well.
    future_days=max(0,29-day)
    feed=0.
    for farm in obs['farms']:
        for row in farm['tiles']:
            for tile_state in row:
                if isinstance(tile_state,dict) and 'animal' in tile_state:
                    feed+=future_days+(0 if tile_state.get('fed_today',False) else 1)

    # Include the feed obligation for our unplaced animals as well.
    owned=totals(obs['private'])
    placement_days=max(0,30-day)
    feed+=placement_days*sum(owned.get(animal,0) for animal in ANIMALS)
    demand['WHEAT']+=feed
    return demand


# -----------------------------------------------------------------------------
# Producer-capacity models
# -----------------------------------------------------------------------------
# These helpers express future crop output in units. Existing crops and a
# hypothetical newly planted seed are measured on the same basis.

def _new_crop_stream_capacity(crop,plant_day,step):
    """Output from a crop planted on plant_day, repeating same crop after final harvest."""
    _,_,events,last=CROPS[crop]
    total=0.
    cycle_day=plant_day
    while cycle_day<30:
        for age,units in events:
            at_day=cycle_day+age
            if at_day<30 and at_day*TURNS_PER_DAY>step:total+=units
        next_cycle=cycle_day+last
        if next_cycle<=cycle_day or next_cycle>=30:break
        cycle_day=next_cycle
    return total

def _current_crop_remaining_capacity(tile_state,day,step):
    """Visible crop output plus assumed same-crop replacement after its final harvest."""
    crop=tile_state['crop'];planted=tile_state['planted_day']
    _,_,events,last=CROPS[crop]
    total=0.
    for age,units in events:
        at_day=planted+age
        if at_day<30 and at_day*TURNS_PER_DAY>step:total+=units
    # Once the current crop finishes, treat the tile as continuing the same production
    # line. If its final harvest is already available/past, assume immediate replant now.
    repeat_day=max(day,planted+last)
    if repeat_day<30:total+=_new_crop_stream_capacity(crop,repeat_day,step)
    return total


# -----------------------------------------------------------------------------
# Signal assembly and ranking
# -----------------------------------------------------------------------------
def production_signals(obs):
    """Rank crops(only) by known remaining demand versus observable own production capacity.

    This is deliberately a signaling service, not a market-inventory forecast:
      - hard demand = Town + unlocked shops + visible herd WHEAT feed;
      - capacity = our crop stock + our ready/current crop yield + our owned seeds;
      - opponent crop production, hidden inventory, and future shops are excluded.

    score = 1-exp(-N), where N is the positive shortage measured in units of one
    additional seed planted now. Higher score means stronger crop-production need.
    """
    # Stage 1: compute remaining hard demand and initialize per-product capacity buckets.
    day=obs['day'];step=day*TURNS_PER_DAY+obs['hour']
    hard_demand=_remaining_hard_demand(obs)
    breakdown={
        p:{'owned_stock':0.,'ready_yield':0.,'current_assets':0.,'owned_producers':0.}
        for p in CROPS
    }

    # Stage 2: finished goods already under our control count directly against future need.
    owned=totals(obs['private'])
    for product in CROPS:
        breakdown[product]['owned_stock']=float(owned.get(product,0))

    # Stage 3: only our planted crops contribute ready yield and remaining output.
    for row in obs['farms'][obs['player']]['tiles']:
        for tile_state in row:
            if not isinstance(tile_state,dict) or 'crop' not in tile_state:continue
            crop=tile_state['crop']
            breakdown[crop]['ready_yield']+=float(tile_state.get('yield_units',0))
            breakdown[crop]['current_assets']+=_current_crop_remaining_capacity(
                tile_state,day,step
            )

    # Stage 4: count our owned seeds as if planted now. Opponent seeds are private.
    seeds=obs['private']['seeds']
    for crop in CROPS:
        count=seeds.get(crop,0)
        if count:
            breakdown[crop]['owned_producers']+=count*_new_crop_stream_capacity(
                crop,day,step
            )
    # Stage 5: convert each crop shortage into new-seed equivalents, then
    # squash it to [0,1] for a comparable need score.
    results=[]
    for product in CROPS:
        capacity=sum(breakdown[product].values())
        demand=hard_demand[product]
        gap=max(0.,demand-capacity)
        one_new=_new_crop_stream_capacity(product,day,step)

        if gap<=0:
            producer_gap=0.
            score=0.
        elif one_new>0:
            producer_gap=gap/one_new
            score=1-math.exp(-producer_gap)
        else:
            producer_gap=math.inf
            score=1.

        results.append({
            'product':product,
            'producer':product,
            'score':score,
            'actionable':one_new>0,
            'hard_demand':demand,
            'capacity':capacity,
            'gap':gap,
            'new_producer_capacity':one_new,
            'producer_equivalent_gap':producer_gap,
            'capacity_breakdown':dict(breakdown[product]),
        })

    # Stage 6: highest production need first; product name is the deterministic tie-breaker.
    results.sort(key=lambda x:(-x['score'],x['product']))
    return results


# =============================================================================
# PRODUCTION PLANNERS
# =============================================================================
def animal_plan(obs,signals):
    """Plan every already-owned animal onto the fixed animal route slots.

    Existing animals already placed on route tiles stay where they are. Every unplaced
    COW/SHEEP/GOOSE in the shed or a worker inventory is assigned to the nearest unlocked,
    unused route slot, regardless of production-signal rank. If a target route tile needs
    cleanup or the correct structure, unit_actions() handles DIG/BUILD/PLACE.

    signals is accepted to keep the planner interface parallel with crop_plan(), but it
    does not gate placement of animals that are already owned.
    """
    f=obs['farms'][obs['player']]
    plan={}

    # Keep animals already placed on route tiles.
    for row in ROUTES:
        for p in row:
            t=tile(f,p)
            if isinstance(t,dict) and t.get('animal'):
                plan[p]=t['animal']

    # Every unlocked unused route point is a possible target. It may be empty, contain
    # a weed/crop, or have the wrong structure; unit_actions() will prepare the tile.
    slots=[
        p for p in ANIMAL_POINTS
        if p not in plan and tile(f,p)!='LOCKED'
    ]
    # Fill quadrant route quotas in policy order before choosing the nearest
    # position within a quadrant. Otherwise SW can consume the final-stage
    # positions while earlier NE requirements are still being completed.
    slots.sort(key=lambda p:(ANIMAL_QUADRANT_ORDER[ANIMAL_QUADRANT[p]],
                             dist(p,nearest_shed(p)),p))

    capacity=min(len(slots),max(0,HERD_LIMIT-len(plan)))
    if capacity<=0:
        return plan

    # totals() = shed + farmer/hand inventories.
    stock=totals(obs['private'])
    selected=[]
    for animal in ANIMALS:
        count=int(stock.get(animal,0))
        take=min(count,capacity-len(selected))
        selected.extend([animal]*take)
        if len(selected)>=capacity:
            break

    # Assign all owned animals that fit to route slots nearest the shed.
    for pos,animal in zip(slots,selected):
        plan[pos]=animal

    return plan


def crop_plan(obs,signals):
    """Plan owned seeds on empty tiles and include crops/weeds for worker maintenance.

    New planting decisions:
      - read seeds already owned in obs['private']['seeds'];
      - find empty, unlocked crop tiles outside ANIMAL_POINTS;
      - if every seed fits, plan every seed;
      - if seeds exceed empty tiles, keep only the highest-ranked seeds according to
        the ordering already supplied by production_signals().

    Then include existing crops and weed tiles outside ANIMAL_POINTS so unit_actions()
    can generate maintenance tasks even when no seeds or empty tiles remain.

    Seed purchasing belongs to market_orders(), not to this planner. The game stores
    seeds in private['seeds'], separately from the ordinary private['shed'] inventory.
    """
    f=obs['farms'][obs['player']]
    seeds={crop:int(obs['private']['seeds'].get(crop,0)) for crop in CROPS}
    plan={}

    # -------------------------------------------------------------------------
    # Stage 1: find truly empty crop-eligible tiles.
    # -------------------------------------------------------------------------
    empty=[
        (x,y)
        for y,row in enumerate(f['tiles'])
        for x,t in enumerate(row)
        if t is None and (x,y) not in ANIMAL_POINTS
    ]
    # Prefer tiles nearer the shed; coordinates provide deterministic tie-breaking.
    empty.sort(key=lambda p:(dist(p,nearest_shed(p)),p[1],p[0]))

    total_seeds=sum(seeds.values())

    # -------------------------------------------------------------------------
    # Stage 2: decide which owned seeds receive tiles.
    # -------------------------------------------------------------------------
    selected=[]

    if total_seeds<=len(empty):
        # Enough land for every seed: ranking is irrelevant because no seed is excluded.
        # CROPS insertion order is used only to make tile assignment deterministic.
        for crop in CROPS:
            selected.extend([crop]*seeds[crop])
    else:
        # Scarce land: production_signals() already returns highest-need products first.
        # Filter that ranking to crop types we actually own, then consume seed counts in
        # rank order until every empty tile has been assigned.
        signal_rank={
            entry['product']:rank
            for rank,entry in enumerate(signals)
            if entry.get('product') in CROPS
        }
        ranked_crops=sorted(
            (crop for crop,count in seeds.items() if count>0),
            key=lambda crop:(signal_rank.get(crop,len(signal_rank)),crop),
        )
        slots=len(empty)
        for crop in ranked_crops:
            take=min(seeds[crop],slots-len(selected))
            selected.extend([crop]*take)
            if len(selected)>=slots:break

    # -------------------------------------------------------------------------
    # Stage 3: pair selected seeds with the empty tiles.
    # -------------------------------------------------------------------------
    for pos,crop in zip(empty,selected):
        plan[pos]=crop

    # -------------------------------------------------------------------------
    # Stage 4: include occupied crop/weed tiles for unit_actions() to inspect.
    # -------------------------------------------------------------------------
    points=[(x,y) for y,row in enumerate(f['tiles']) for x,t in enumerate(row)
            if t!='LOCKED' and (x,y) not in ANIMAL_POINTS]
    points.sort(key=lambda p:(dist(p,nearest_shed(p)),p[1],p[0]))
    for p in points:
        t=tile(f,p)
        if not isinstance(t,dict):continue
        if 'crop' in t:
            plan[p]=t['crop']
        elif t.get('kind')=='WEED':
            # The crop name is a placeholder: unit_actions() issues DIG for weeds
            # before reading it. Actual planting is planned once the tile is empty.
            plan[p]='WHEAT'
    return plan

def start_plan(obs,cash=None,slots=10):
    """Return unmet day-0 opening targets in priority order, up to `slots` orders.

    Canonical sequence:
      BUY_SEED WHEAT 13
      BUY_ANIMAL COW 2
      BUY_SEED CARROT 1
      BUY_SEED STRAWBERRY 4
      3 x HIRE
      BUY_SEED MELON 1
      BUY_ANIMAL SHEEP 1
      BUY_ANIMAL GOOSE 1
      2 x HIRE

    """
    f=obs['farms'][obs['player']];p=obs['private'];day=obs['day']
    if day!=0 or slots<=0:return []

    # The first call can submit only ten of the twelve commands. On each later call,
    # count purchased assets wherever workers may have moved them before retrying.
    owned={c:int(p['seeds'].get(c,0)) for c in CROPS}
    stock=totals(p)
    owned.update({a:int(stock.get(a,0)) for a in ANIMALS})
    for row in f['tiles']:
        for t in row:
            if not isinstance(t,dict):continue
            crop=t.get('crop');animal=t.get('animal')
            if crop in CROPS:owned[crop]+=1
            if animal in ANIMALS:owned[animal]+=1

    opening=[
        ('BUY_SEED','WHEAT',13),
        ('BUY_ANIMAL','COW',2),
        ('BUY_SEED','CARROT',1),
        ('BUY_SEED','STRAWBERRY',4),
        ('HIRE',None,3),
        ('BUY_SEED','MELON',1),
        ('BUY_ANIMAL','SHEEP',1),
        ('BUY_ANIMAL','GOOSE',1),
        ('HIRE',None,5),
    ]
    if cash is None:cash=f['money']
    fib=[1,1]
    for _ in range(14):fib.append(fib[-1]+fib[-2])
    orders=[];planned_hires=0
    for op,item,target in opening:
        if len(orders)>=slots:break
        if op=='HIRE':
            missing=max(0,target-len(f['hands'])-planned_hires)
            for _ in range(missing):
                if len(orders)>=slots:break
                cost=fib[f['hires_today']+planned_hires]
                if cash<cost:break
                orders.append(['HIRE']);cash-=cost;planned_hires+=1
            continue
        missing=max(0,target-owned[item])
        cost=CROPS[item][0] if op=='BUY_SEED' else ANIMALS[item][0]
        n=min(missing,int(max(0,cash)//cost))
        if n>0:
            orders.append([op,item,n]);cash-=n*cost;owned[item]+=n
    return orders


class AnimalBuyingPolicy:
    """Rebuild the ordered animal requirements from the current observation.

    Each requirement stores its increment and cumulative species targets. A later
    requirement is considered only after every earlier cumulative target is met.
    This needs no cross-turn memory: shed, tile, and worker-carried animals are
    recounted each turn, so failed orders and escapes reopen their unmet stage.
    """

    # (first buying day, last buying day, quadrant, default increment, shop day,
    #  animal units if that newly opened shop needs an animal product)
    STAGES=(
        (1,2,'NW',EARLY_NW_ANIMAL_TARGETS,None,0),
        (3,4,'NE',{'COW':1,'SHEEP':1},3,2),
        (6,7,'NE',{'COW':1,'GOOSE':1},6,2),
        (9,10,'NE',{'COW':1,'GOOSE':1,'SHEEP':1},9,3),
        (12,13,'SW',{'GOOSE':1,'SHEEP':1},12,2),
        (15,17,'SW',{'GOOSE':1,'COW':1},15,2),
    )
    SHOP_ANIMAL={
        'YARN_STORE':'SHEEP',
        'PIZZA_SHOP':'COW',
        'ICE_CREAM_SHOP':'COW',
        'SMOOTHIE_SHOP':'COW',
        'BAKERY':'GOOSE',
        'BRUNCH_SPOT':'GOOSE',
    }

    def __init__(self,obs):
        self.obs=obs
        self.farm=obs['farms'][obs['player']]
        self.day=obs['day']
        private=obs['private']
        ############# all owned animals
        self.owned={a:int(private['shed'].get(a,0)) for a in ANIMALS}
        for row in self.farm['tiles']:
            for t in row:
                if isinstance(t,dict) and t.get('animal') in ANIMALS:
                    self.owned[t['animal']]+=1
        for inv in private['inventories']:
            for animal in ANIMALS:
                self.owned[animal]+=int(inv.get(animal,0))
        ##############

        unlocked=sum(tile(self.farm,p)!='LOCKED' for p in ANIMAL_POINTS)
        self.space=max(0,min(HERD_LIMIT,unlocked)-sum(self.owned.values()))
        self.requirements=[]
        cumulative={a:0 for a in ANIMALS}
        for first,last,region,default,shop_day,shop_units in self.STAGES:
            if first>self.day:break
            increment=self._increment(default,shop_day,shop_units)
            for animal,n in increment.items():cumulative[animal]+=n
            self.requirements.append(dict(first=first,last=last,region=region,
                                          increment=increment,target=dict(cumulative)))

    def _increment(self,default,shop_day,shop_units):
        # The D1-D2 opening target is fixed; later stages use their new shop.
        if shop_day is None:
            return dict(default)

        # Each stage starts on a shop-opening day (D3, D6, ...).
        shops=self.obs.get('town',{}).get('unlocked_shops',[])
        index=shop_day//3-1
        shop=shops[index] if index<len(shops) else None

        # Shop type selects the species; stage size selects the animal count.
        # A YARN_STORE's two-wool demand still represents just one shop.
        animal=self.SHOP_ANIMAL.get(shop)
        return {animal:shop_units} if animal else dict(default)

    def _region_free_slots(self,region):
        # Only designated animal routes in this quadrant can hold the herd.
        free=0
        for x,y in ANIMAL_POINTS:
            if ANIMAL_QUADRANT[(x,y)]!=region:
                continue
            t=tile(self.farm,(x,y))

            # An unlocked route is available if it has no placed animal.
            if t!='LOCKED' and not (isinstance(t,dict) and t.get('animal') in ANIMALS):
                free+=1
        return free

    def buy_orders(self,cash,slots):
        # Buy only during a stage's allowed days and when market slots remain.
        if slots<=0 or not any(s[0]<=self.day<=s[1] for s in self.STAGES):
            return [],cash

        # Track orders against the current owned count, cash, and route capacity.
        orders=[]
        budget=2
        planned=dict(self.owned)
        space=self.space
        region_space={r:self._region_free_slots(r) for r in ('NW','NE','SW')}
        regions=('NW','NE','SW')

        # Work through cumulative stage targets in order. An escape can reopen
        # an earlier target, which must be filled before later requirements.
        for requirement in self.requirements:
            target=requirement['target']
            if all(planned[a]>=target[a] for a in ANIMALS):
                continue

            # Prefer this stage's quadrant. Use an earlier free route to
            # replace an escape if the stage's quadrant is already full.
            region=requirement['region']
            available=tuple(r for r in reversed(regions[:regions.index(region)+1])
                            if region_space[r]>0)
            if not available:
                break

            # Buy this stage's species first, then fill any other species gap
            # needed to finish its cumulative target.
            species=tuple(requirement['increment'])+tuple(
                a for a in ANIMALS if a not in requirement['increment'])
            for animal in species:
                if len(orders)>=slots:
                    break
                gap=max(0,target[animal]-planned[animal])
                if gap<=0:
                    continue

                # Limit this purchase by cash reserve, available routes, and
                # the two-animal per-turn budget.
                cost=ANIMALS[animal][0]
                affordable=int(max(0,cash-PROCUREMENT_CASH_RESERVE)//cost)
                selected=next((r for r in available if region_space[r]>0),None)
                if selected is None:
                    break
                n=min(gap,space,region_space[selected],budget,affordable)
                if n<=0:
                    continue

                # Reserve the money and capacity for subsequent orders.
                orders.append(['BUY_ANIMAL',animal,n])
                cash-=n*cost
                planned[animal]+=n
                space-=n
                budget-=n
                region_space[selected]-=n

            # Never use this turn's remaining budget for a later requirement
            # while the current requirement still has a species deficit.
            if any(planned[a]<target[a] for a in ANIMALS):
                break
        return orders,cash


def melon_seed_order(obs,cash,orders):
    """Buy one melon seed on even days through the last melon buying day.

    Hour 2 avoids the start-of-day hire orders. On that turn a new seed is due
    even if an older seed remains in storage. Later turns use a held seed or a
    melon planted today to avoid repeating the order.
    """
    day=obs['day']
    if day==0 or day%2 or day>LATEST_BUY_DAY['MELON'] or obs['hour']<2 or len(orders)>=10:
        return None
    if cash<CROPS['MELON'][0]:
        return None
    if any(order[:2]==['BUY_SEED','MELON'] for order in orders):
        return None
    if obs['hour']>2 and obs['private']['seeds'].get('MELON',0):
        return None
    farm=obs['farms'][obs['player']]
    if obs['hour']>2 and any(isinstance(t,dict) and t.get('crop')=='MELON' and
           t.get('planted_day')==day for row in farm['tiles'] for t in row):
        return None
    return ['BUY_SEED','MELON',1]


def market_orders(obs,signals,actions):
    # Build the rule-based market order list (maximum 10 orders). First simulate the shed
    # after current worker PICKUP/DROP/PLACE actions, then emit SELL orders for available
    # products while reserving wheat for the herd and keeping fertilizer at a
    # single target shared by the SELL and BUY stages.
    # Day 0/hour 0 is a fixed 10-order opening. Otherwise, remaining slots are considered in
    # this order after SELL: short-horizon BUY_PRODUCT WHEAT, HIRE, BUY_LAND,
    # staged BUY_ANIMAL (at most two animal units in this turn), ranked
    # signal-driven BUY_SEED, then BUY_PRODUCT FERTILIZER.
    # Feed WHEAT is intentionally separate from production_signals(): seeds solve future
    # production capacity, while animals need consumable WHEAT before those crops mature.
    # Dynamic producer procurement starts after day 0 so the fixed opening remains authoritative.
    # Every stage checks cash and
    # order capacity; earlier SELL/HIRE orders can consume slots needed by later purchases,
    # and the final `orders[:10]` enforces the engine limit defensively.
    # -------------------------------------------------------------------------
    # SETUP — snapshot cash/inventory and predict what the shed contains after
    # this turn's worker actions, because market orders execute afterward.
    # -------------------------------------------------------------------------
    f=obs['farms'][obs['player']];p=obs['private'];day=obs['day'];hour=obs['hour'];prices=obs['market']['prices']
    cash=f['money'];orders=[]
    held,sell_total=post_action_market_inventory(obs,actions)
    live=sum(1 for row in f['tiles'] for t in row if isinstance(t,dict) and 'animal' in t)# animal numbers
    fert_need=sum(1 for row in f['tiles'] for t in row
                  if isinstance(t,dict) and 'crop' in t)
    # Tomatoes and strawberries cannot use fertilizer early in this opening.
    # Keep that early fertilizer liquid so seed and livestock purchases have cash.
    fert_target=min(24,fert_need+3) if fert_need and 9<=day<29 else 0
    fert_owned=sell_total.get('FERTILIZER',0)
    # -------------------------------------------------------------------------
    # 1. SELL — liquidate immediately sellable goods first, while keeping the
    # WHEAT reserve and fertilizer target. Sale proceeds fund later priorities.
    # -------------------------------------------------------------------------
    wheat_buffer=2 if live<4 else 4 if live<=8 else 7 #wheat reserve at each turn
    reserve_wheat=min(live+2,wheat_buffer)
    for c in SELLABLE_PRODUCTS:
        n=held.get(c,0)
        if c=='WHEAT':n=min(n,max(0,sell_total.get(c,0)-reserve_wheat))
        if c=='FERTILIZER':n=min(n,max(0,fert_owned-fert_target))
        if n:
            orders.append(['SELL',c,n]);cash+=n*max(1,prices[c]*.8)
    # -------------------------------------------------------------------------
    # FIRST-TURN OVERRIDE — the fixed opening plan is authoritative. Any SELL
    # orders computed above are intentionally discarded on day 0 / hour 0.
    # -------------------------------------------------------------------------
    if day==0 and hour==0:
        return start_plan(obs,cash,10)

    # -------------------------------------------------------------------------
    # 2. BUY FEED WHEAT — use the same herd-aware reserve as the SELL gate.
    # This is a short-horizon inventory requirement, not a producer-capacity signal.
    # Place it before HIRE/land/producer buying so a full order list cannot starve
    # existing animals while spending on expansion.
    # -------------------------------------------------------------------------
    wheat_owned_after_actions=sell_total.get('WHEAT',0)
    if day<29 and wheat_owned_after_actions<reserve_wheat and len(orders)<10:
        shortage=reserve_wheat-wheat_owned_after_actions
        unit_cost=prices['WHEAT']+3
        n=min(shortage,int(max(0,cash-10)//unit_cost))
        if n>0:
            orders.append(['BUY_PRODUCT','WHEAT',n])
            cash-=n*unit_cost

    # -------------------------------------------------------------------------
    # DAY-0 OPENING REMAINDER — reconcile all opening targets against current
    # ownership, then submit what did not fit or execute on earlier turns.
    # -------------------------------------------------------------------------
    # start_plan() owns the fixed opening policy; market_orders() only appends the
    # remaining day-0 opening purchases into whatever order capacity is still available.
    if day==0 and len(orders)<10:
        opening_orders=start_plan(obs,cash,10-len(orders))
        fib=[1,1]
        for _ in range(14):fib.append(fib[-1]+fib[-2])
        opening_hires=0
        for order in opening_orders:
            orders.append(order)
            # start_plan() returns commands only; market_orders() maintains its own running
            # cash estimate so later purchases in this same turn cannot overspend it.
            if order[0]=='BUY_SEED':cash-=order[2]*CROPS[order[1]][0]
            elif order[0]=='BUY_ANIMAL':cash-=order[2]*ANIMALS[order[1]][0]
            elif order[0]=='HIRE':
                cash-=fib[f['hires_today']+opening_hires]
                opening_hires+=1

    # -------------------------------------------------------------------------
    # 3. HIRE — fill the current hand target before spending on land or producers.
    # Existing timing, Fibonacci cost, and cash-buffer rules are preserved.
    # -------------------------------------------------------------------------
    desired_hands=5 if len(f['unlocked_quadrants'])==1 else 8 if len(f['unlocked_quadrants'])==2 else 11
    if day<3:desired_hands=5
    elif day==29 and OPP_STYLE=='V16':desired_hands=min(desired_hands,V16_FINAL_HANDS)
    if hour<4:
        planned_hires=sum(order[0]=='HIRE' for order in orders)
        hires=f['hires_today']+planned_hires;fib=[1,1]
        for _ in range(14):fib.append(fib[-1]+fib[-2])
        for _ in range(max(0,desired_hands-len(f['hands'])-planned_hires)):
            cost=fib[hires]
            if len(orders)>=10 or cash<cost+20:break
            orders.append(['HIRE']);cash-=cost;hires+=1
    # -------------------------------------------------------------------------
    # 4. BUY LAND — expand only when free tiles fall below the capacity threshold.
    # Land gets priority over new seeds/animals once the expansion rule triggers.
    # -------------------------------------------------------------------------
    lands=len(f['unlocked_quadrants'])
    empty_tiles=sum(1 for row in f['tiles'] for t in row if t is None)
    # (empty-tile threshold, land cost, latest buy day)
    # The 4th quadrant is not worth buying after day 24.
    land_policy={1:(2,1000,29),2:(2,2000,29),3:(2,4000,24)}
    if lands in land_policy:
        empty_threshold,landcost,latest_buy_day=land_policy[lands]
        if day<=latest_buy_day and empty_tiles<=empty_threshold and cash>=landcost and len(orders)<10:
            orders.append(['BUY_LAND']);cash-=landcost
    # -------------------------------------------------------------------------
    # 5. BUY ANIMALS — complete cumulative NW, NE, and SW requirements in order.
    # The planner reads the opening shop for each stage and retries unfinished
    # earlier requirements before later ones on each designated buying day.
    # -------------------------------------------------------------------------
    if day>0 and len(orders)<10:
        animal_orders,cash=AnimalBuyingPolicy(obs).buy_orders(cash,10-len(orders))
        orders.extend(animal_orders)
    # -------------------------------------------------------------------------
    # 6. BUY SEEDS — keep MELON on a two-day cadence through its last buying day,
    # then preserve production_signals() rank for the other crops. Use the cash
    # and order slots remaining after animal purchases.
    # -------------------------------------------------------------------------
    if day>0 and len(orders)<10:
        melon_order=melon_seed_order(obs,cash,orders)
        if melon_order:
            orders.append(melon_order)
            cash-=CROPS['MELON'][0]
        crop_slots=sum(
            1 for y,row in enumerate(f['tiles']) for x,t in enumerate(row)
            if t is None and (x,y) not in ANIMAL_POINTS
        )
        crop_slots=max(0,crop_slots-sum(int(p['seeds'].get(c,0)) for c in CROPS))
        for signal in signals:
            if len(orders)>=10:break
            if not signal.get('actionable'):continue
            producer=signal['producer']
            if producer not in CROPS:continue
            if producer=='MELON':continue
            if day>LATEST_BUY_DAY.get(producer,29):continue
            gap=signal.get('producer_equivalent_gap',0.)
            if not math.isfinite(gap):continue
            needed=int(math.floor(gap))
            if needed<=0:continue
            spendable=max(0,cash-PROCUREMENT_CASH_RESERVE)
            cost=CROPS[producer][0]
            affordable=int(spendable//cost)
            n=min(needed,crop_slots,affordable)
            if n<=0:continue
            orders.append(['BUY_SEED',producer,n])
            cash-=n*cost
            crop_slots-=n
    # -------------------------------------------------------------------------
    # 7. BUY FERTILIZER — last priority. Buy only the shortage against the same
    # target used by SELL, if cash and order capacity remain.
    # -------------------------------------------------------------------------
    # The PPO worker policy decides where and when to apply fertilizer.
    if fert_owned<fert_target and len(orders)<10:
        n=min(fert_target-fert_owned,int(max(0,cash-100)//(prices['FERTILIZER']+3)))
        if n:orders.append(['BUY_PRODUCT','FERTILIZER',n])
    # -------------------------------------------------------------------------
    # FINALIZE — enforce the engine's maximum of 10 market orders.
    # -------------------------------------------------------------------------
    return orders[:10]

def agent(obs):
    global OPP_STYLE

    # -------------------------------------------------------------------------
    # 1. GAME / OPPONENT STATE
    # -------------------------------------------------------------------------
    # Only the V16 archetype still changes active policy behavior. All other
    # opponents share the normal path, so avoid carrying dead trader subtypes.
    if obs['day']==0 and obs['hour']==0:
        OPP_STYLE=None
    elif OPP_STYLE is None and obs['day']==0 and obs['hour']>0:
        other=obs['farms'][1-obs['player']]
        OPP_STYLE='V16' if other['hires_today']==5 and 20<=other['money']<120 else 'NORMAL'

    # -------------------------------------------------------------------------
    # 2. PLAN THIS TURN
    # -------------------------------------------------------------------------
    environment_signals=production_signals(obs)
    animal=animal_plan(obs,environment_signals)
    crops=crop_plan(obs,environment_signals)

    # -------------------------------------------------------------------------
    # 3. EXECUTE WORKERS, THEN PLAN MARKET ORDERS
    # -------------------------------------------------------------------------
    # Market planning must follow worker planning because it simulates this
    # turn's PICKUP/DROP/PLACE/consumption actions before deciding what can sell.
    actions=unit_actions(obs,animal,crops)
    market=market_orders(obs,environment_signals,actions)

    # -------------------------------------------------------------------------
    # 4. RETURN ENVIRONMENT ACTION
    # -------------------------------------------------------------------------
    return dict(farmer=actions[0],hands=actions[1:],market=market)
'''

_ACTOR_B64 = (
    "..."
    "q8ID9ZSMMPR2Gih9KqZLk0PAVbMQ/gfO2SNA"
)
_ACTOR_SHA256 = "fb411bf99f56e5747a756e9472eed908715bda46cd71e63bb0a34b887293ea65"

def _load_module(name, source):
    module = types.ModuleType(name)
    sys.modules[name] = module
    exec(compile(source, name + '.py', 'exec'), module.__dict__)
    return module

worker_reward = _load_module('worker_reward', _REWARD_SOURCE)
worker_policy = _load_module('worker_policy', _POLICY_SOURCE)
v25_rl = _load_module('v25_rl_runtime', _PLANNER_SOURCE)
actor_bytes = zlib.decompress(base64.b64decode(_ACTOR_B64))
assert hashlib.sha256(actor_bytes).hexdigest() == _ACTOR_SHA256
model = worker_policy.ActorModel(actor_bytes, len(worker_policy.CANDIDATE_FEATURE_NAMES))
policy = worker_policy.WorkerPolicy(v25_rl, model, None, deterministic=True, collect=False)

def agent(obs):
    v25_rl.unit_actions = policy.unit_actions
    return v25_rl.agent(obs)


Writing main.py


## 3. Check the entrypoint and action contract

This catches the filename/function mismatch that the tutorial's in-memory callable test misses.

In [3]:
"""Notebook smoke-test cell: use a complete initial market and both farms."""
import ast
import copy
import importlib.util
import json
from pathlib import Path

main_path = Path("main.py")
assert main_path.is_file(), "main.py was not created"
tree = ast.parse(main_path.read_text(encoding="utf-8"), filename="main.py")
assert any(isinstance(n, ast.FunctionDef) and n.name == "agent" for n in tree.body)
spec = importlib.util.spec_from_file_location("submission_agent", main_path)
submission_agent = importlib.util.module_from_spec(spec)
spec.loader.exec_module(submission_agent)
assert callable(submission_agent.agent)

base_prices = {
    "WHEAT": 25, "CARROT": 35, "TOMATO": 60, "STRAWBERRY": 120,
    "MELON": 250, "EGG": 50, "MILK": 160, "WOOL": 200, "FERTILIZER": 100,
}
farm = {
    "money": 3000,
    "tiles": [[None if x < 5 and y < 5 else "LOCKED" for x in range(10)] for y in range(10)],
    "farmer": [4, 4], "hands": [], "unlocked_quadrants": ["NW"], "hires_today": 0,
}
dummy_obs = {
    "player": 0, "day": 0, "hour": 0,
    "farms": [copy.deepcopy(farm), copy.deepcopy(farm)],
    "private": {"shed": {}, "seeds": {}, "inventories": [{}]},
    "market": {"inventory": {item: 10000 for item in base_prices}, "prices": base_prices},
    "town": {"unlocked_shops": []},
}
for seat in (0, 1):
    obs = copy.deepcopy(dummy_obs)
    obs["player"] = seat
    action = submission_agent.agent(obs)
    assert set(action) == {"farmer", "hands", "market"}
    assert isinstance(action["farmer"], list) and action["farmer"]
    assert isinstance(action["hands"], list) and len(action["hands"]) == 0
    assert isinstance(action["market"], list) and len(action["market"]) <= 10
    json.dumps(action)
print("Entrypoint and JSON action contract, both seats: OK")
print(action)


Entrypoint and JSON action contract, both seats: OK
{'farmer': ['PASS'], 'hands': [], 'market': [['BUY_SEED', 'WHEAT', 13], ['BUY_ANIMAL', 'COW', 2], ['BUY_SEED', 'CARROT', 1], ['BUY_SEED', 'STRAWBERRY', 4], ['HIRE'], ['HIRE'], ['HIRE'], ['BUY_SEED', 'MELON', 1], ['BUY_ANIMAL', 'SHEEP', 1], ['BUY_ANIMAL', 'GOOSE', 1]]}


## 4. Run full file-loader validation games

The first match mirrors Kaggle's self-play Validation Episode. The other matches check two built-in opponents. All games run the **`main.py` path**, not an in-memory function.

In [4]:
from kaggle_environments import make

opponents = ["main.py", "starter", "random"]
for index, opponent in enumerate(opponents):
    env = make(
        "kaggriculture",
        configuration={"episodeSteps": 720, "seed": 20260822 + index},
        debug=True,
    )
    env.run(["main.py", opponent])
    final = env.steps[-1]
    statuses = [state.status for state in final]
    rewards = [state.reward for state in final]
    assert statuses == ["DONE", "DONE"], (opponent, statuses)
    print(f"vs {opponent:8s}: statuses={statuses}, rewards={rewards}")

print("Full 720-turn file-loader validation: OK")

vs main.py : statuses=['DONE', 'DONE'], rewards=[55976.0, 53021.0]
vs starter : statuses=['DONE', 'DONE'], rewards=[50226.0, 3682.0]
vs random  : statuses=['DONE', 'DONE'], rewards=[58684.0, 0.0]
Full 720-turn file-loader validation: OK


## 5. Build and verify the submission archive

The member name must be exactly `main.py`, with no enclosing directory.

In [5]:
import hashlib
import tarfile
from pathlib import Path

archive_path = Path("submission.tar.gz")
with tarfile.open(archive_path, "w:gz") as archive:
    archive.add("main.py", arcname="main.py", recursive=False)

with tarfile.open(archive_path, "r:gz") as archive:
    members = archive.getnames()
    assert members == ["main.py"], members
    archived_source = archive.extractfile("main.py").read()

assert archived_source == Path("main.py").read_bytes()
size_mib = archive_path.stat().st_size / (1024 * 1024)
assert size_mib < 100, f"Archive is too large: {size_mib:.2f} MiB"
sha256 = hashlib.sha256(archive_path.read_bytes()).hexdigest()

print(f"Created: {archive_path.resolve()}")
print(f"Members: {members}")
print(f"Size: {size_mib:.4f} MiB")
print(f"SHA-256: {sha256}")

Created: /kaggle/working/submission.tar.gz
Members: ['main.py']
Size: 0.1149 MiB
SHA-256: f1ff2f7330580c3bd7f34475124d95d098ad265b8bdde1453d89642d24403aa4


## 6. Submit

1. Attach the **Kaggriculture** competition to this notebook and accept the competition rules.
2. Run all cells and save a successful notebook version.
3. Click **Submit to competition** and select the notebook output `submission.tar.gz`.

CLI equivalent:

```bash
kaggle competitions submit kaggriculture \
  -k YOUR_USERNAME/YOUR_NOTEBOOK_SLUG \
  -f submission.tar.gz \
  -v NOTEBOOK_VERSION \
  -m "mixed crop submission-ready agent"
```

After submission, confirm that the self-play Validation Episode finishes without an `Error` status.